## RQ1: Prevalence, Characterization, and Thematic Nature of Racialized Discourse
"What is the prevalence of racialized or culturally charged discourse in public pool reviews, and how is it characterized in terms of reviewer behavior, linguistic style, thematic content, and spatiotemporal distribution?"

Prevalence: How common is it? (Simple proportion)

Reviewer Characterization: Pseudonyms? Profile pictures? Reviewing history? (Behavioral signals)

Linguistic Characterization: Explicit vs. implicit (e.g., dog-whistles like "certain clientele," "atmosphere changed")?

Thematic Characterization: What are people actually talking about? Hygiene, safety, culture, language, crowding? (This is your thematic analysis, which explains the what).

Spatiotemporal Characterization: Concentrated in East/West? Urban/Rural? Has it increased over time? Do reviews get "cleaned up" via editing?

Why one RQ? Because all these elements answer the same overarching question: "What does this phenomenon look like in the wild?" Separating themes into RQ2 creates an artificial break. The themes are a key part of the characterization.

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os
import matplotlib.pyplot as plt
import functools

## Read data and save plots

In [ ]:
def read_parquet_with_checks(file_path):
    print(f"File size: {os.path.getsize(file_path)} bytes")

    # Try to read just metadata
    try:
        metadata = pq.read_metadata(file_path)
        print("Metadata read successfully")
        print(f"Number of rows: {metadata.num_rows}")
        print(f"Number of columns: {metadata.num_columns}")
    except Exception as e:
        print(f"Error reading metadata: {e}")

    try:
      df= pd.read_parquet(
        file_path,
        engine='fastparquet'  # Instead of pyarrow
        )
    except Exception as e:
      print(f"Fastparquet error: {e}")    
    return df  

In [ ]:
import matplotlib.pyplot as plt
import os
import functools

# Create folder
save_dir = "/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/visuals"
os.makedirs(save_dir, exist_ok=True)

# Store the original savefig function from Figure class
original_fig_savefig = plt.Figure.savefig

@functools.wraps(original_fig_savefig)
def new_savefig(self, fname, *args, **kwargs):
    # Only modify if fname is a string (not a file object or figure)
    if isinstance(fname, str):
        # If it's not an absolute path, prepend our directory
        if not os.path.isabs(fname):
            fname = os.path.join(save_dir, fname)
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(fname), exist_ok=True)
    
    # Call original savefig
    return original_fig_savefig(self, fname, *args, **kwargs)

# Replace the Figure class's savefig method
plt.Figure.savefig = new_savefig

# Also patch pyplot.savefig
original_pyplot_savefig = plt.savefig

@functools.wraps(original_pyplot_savefig)
def new_pyplot_savefig(fname, *args, **kwargs):
    # Only modify if fname is a string
    if isinstance(fname, str):
        # If it's not an absolute path, prepend our directory
        if not os.path.isabs(fname):
            fname = os.path.join(save_dir, fname)
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(fname), exist_ok=True)
    
    # Call original savefig
    return original_pyplot_savefig(fname, *args, **kwargs)

plt.savefig = new_pyplot_savefig

print(f"✓ All figures will be saved to: {os.path.abspath(save_dir)}")

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("/Users/i2scg/Documents/PHD/sm_project/btw25_geometrie_wahlkreise_shp_geo/btw25_geometrie_wahlkreise_shp_geo.shp")

#rename columns
gdf.rename(columns={'WKR_NR':'constituency_nr','WKR_NAME':'constituency' , 'LAND_NAME':'state', 'LAND_NR':'state_nr'}, inplace=True)
gdf.info()

In [ ]:
#demographic data
demographic_data = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/demographics.csv")
#rename columns
# Suggested new column names:
column_rename_dict = {
    # Original: New Name
    'municipalities_31.12.2023(number)': 'municipalities_count',
    'area_31.12.2023(kmsq)': 'area_km2',
    'population_31.12.2023 - total(in 1000)': 'population_total_thousands',
    'population_31.12.2023_germans(in 1000)': 'population_german_thousands',
    'population_31.12.2023_foreign_nationals(%)': 'population_foreign_percent',
    'population_density_31.12.2023 (inhabitants_per_km²)': 'population_density_per_km2',
    'population increase (+) or decrease (-) in 2023 - birth balance (per 1000 inhabitants)': 'birth_rate_per_1000',
    'population increase (+) or decrease (-) in 2022 - migration balance (per 1000 inhabitants)': 'migration_rate_per_1000_2022',
    'age from ... to ... years on 31.12.2023 - under 18 (%)': 'age_under_18_percent',
    'age from ... to ... years on 31.12.2023 - 18-24 (%)': 'age_18_24_percent',
    'age from ... to ... years on 31.12.2023 - 25-34 (%)': 'age_25_34_percent',
    'age from ... to ... years on 31.12.2023 - 35-59 (%)': 'age_35_59_percent',
    'age from ... to ... years on 31.12.2023 - 60-74 (%)': 'age_60_74_percent',
    'age from ... to ... years on 31.12.2023 - 75 and over (%)': 'age_75_plus_percent'
}

demographic_data = demographic_data.rename(columns=column_rename_dict)

In [ ]:

#voter_data 
voter_data = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/voters_share.csv")

demographic_voters_data = pd.merge(demographic_data, voter_data, on=['constituency_nr', 'constituency', 'state'], how='inner')
#make column names lowercase
demographic_voters_data.columns = [col.lower() for col in demographic_voters_data.columns]
demographic_voters_data.info()
# save merged data
demographic_voters_data.to_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/demographics_voters_data.csv", index=False)

## Classified reviews

In [ ]:
all_reviews_path = read_parquet_with_checks("//Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/all_collected_text_reviews_25112025.parquet")
all_reviews_path.info()

In [ ]:
analysis_df = read_parquet_with_checks("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/merged_700k_classified_with_ethnicity_13012026.parquet")
analysis_df.info()# Usage examples

In [ ]:
analysis_df = analysis_df.merge(all_reviews_path[['review_id', 'profile_pic']], on='review_id', how='left')
analysis_df.info()

In [ ]:
racial_reviews = read_parquet_with_checks("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/final_racialreviews_from_700k_12012026.parquet")
racial_reviews.info()

In [ ]:
#drop columns that are in demographic_voters_data
racial_reviews = racial_reviews.drop(columns=demographic_voters_data.columns.intersection(racial_reviews.columns).tolist())
#drop these columns too
#racial_reviews = racial_reviews.drop(columns=['land_code','constituency_code', 'constituency_number', 'total_population', 'german_population', 'foreign_population'])

racial_reviews.info()

In [ ]:
racial_themes_df = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/racial_theme_analysis_20260106_221456/theme_classification_20260107_185342/theme_classified_reviews_claude-3.5-sonnet.csv")
racial_themes_df.info()

## merging with shape files

### merging the main data of reviews first

In [ ]:
# Create points GeoDataFrame with explicit CRS
analysis_gdf = gpd.GeoDataFrame(
    analysis_df, 
    geometry=gpd.points_from_xy(analysis_df.longitude, analysis_df.latitude), 
    crs="EPSG:4326"  # WGS84 (lat/lon)
)

print("Points CRS:", analysis_gdf.crs)
print("Points bounds:", analysis_gdf.total_bounds)

# Ensure both have the same CRS
if gdf.crs != analysis_gdf.crs:
    print(f"CRS mismatch! Converting points to shapefile CRS: {gdf.crs}")
    analysis_gdf = analysis_gdf.to_crs(gdf.crs)

In [ ]:
# First try intersects instead of within
analysis_gdf = gpd.sjoin(analysis_gdf, gdf, how='left', predicate='intersects')

# If there are still issues, use this comprehensive approach:
if analysis_gdf['index_right'].isna().sum() > 0:
    # For remaining points, use nearest neighbor
    from scipy.spatial import cKDTree
    
    # Get centroids and build KDTree
    centroids = np.array([[p.x, p.y] for p in gdf.geometry.centroid])
    tree = cKDTree(centroids)
    
    non_matching_idx = analysis_gdf['index_right'].isna()
    non_matching_points = np.array([[p.x, p.y] for p in analysis_gdf.loc[non_matching_idx, 'geometry']])
    
    if len(non_matching_points) > 0:
        distances, indices = tree.query(non_matching_points)
        
        for i, idx in enumerate(analysis_gdf[non_matching_idx].index):
            nearest = gdf.iloc[indices[i]]
            analysis_gdf.loc[idx, ['constituency_nr', 'constituency', 'state', 'state_nr']] = \
                nearest[['constituency_nr', 'constituency', 'state', 'state_nr']].values

In [ ]:
#merge with demographic_voters_data
analysis_gdf = analysis_gdf.merge(demographic_voters_data, on=['constituency_nr', 'constituency', 'state'], how='left')

In [ ]:
analysis_gdf.info()

In [ ]:
analysis_gdf.to_parquet("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/merged_700k_classified_with_ethnicity_14012026.parquet")

### merging racial reviews

In [ ]:
# Create points GeoDataFrame with explicit CRS
racial_reviews_gdf = gpd.GeoDataFrame(
    racial_reviews, 
    geometry=gpd.points_from_xy(racial_reviews.longitude, racial_reviews.latitude), 
    crs="EPSG:4326"  # WGS84 (lat/lon)
)

print("Points CRS:", racial_reviews_gdf.crs)
print("Points bounds:", racial_reviews_gdf.total_bounds)

# Ensure both have the same CRS
if gdf.crs != racial_reviews_gdf.crs:
    print(f"CRS mismatch! Converting points to shapefile CRS: {gdf.crs}")
    racial_reviews_gdf = racial_reviews_gdf.to_crs(gdf.crs)

In [ ]:
# First try intersects instead of within
racial_reviews_gdf = gpd.sjoin(racial_reviews_gdf, gdf, how='left', predicate='intersects')

# If there are still issues, use this comprehensive approach:
if racial_reviews_gdf['index_right'].isna().sum() > 0:
    # For remaining points, use nearest neighbor
    from scipy.spatial import cKDTree
    
    # Get centroids and build KDTree
    centroids = np.array([[p.x, p.y] for p in gdf.geometry.centroid])
    tree = cKDTree(centroids)
    
    non_matching_idx = racial_reviews_gdf['index_right'].isna()
    non_matching_points = np.array([[p.x, p.y] for p in racial_reviews_gdf.loc[non_matching_idx, 'geometry']])
    
    if len(non_matching_points) > 0:
        distances, indices = tree.query(non_matching_points)
        
        for i, idx in enumerate(racial_reviews_gdf[non_matching_idx].index):
            nearest = gdf.iloc[indices[i]]
            racial_reviews_gdf.loc[idx, ['constituency_nr', 'constituency', 'state', 'state_nr']] = \
                nearest[['constituency_nr', 'constituency', 'state', 'state_nr']].values

In [ ]:
#merge with demographic_voters_data
racial_reviews_gdf = racial_reviews_gdf.merge(demographic_voters_data, on=['constituency_nr', 'constituency', 'state'], how='left')

In [ ]:
racial_reviews_gdf.info()

In [ ]:
racial_reviews_gdf.to_parquet("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/final_racialreviews_from_700k_15012026.parquet")

## Add  rerun reviews for themes (run already)

In [ ]:
# reclassifed_themes_df = pd.read_csv("//Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/racial_theme_analysis_20260106_221456/theme_classification_20260112_150447/theme_classified_reviews_claude-3.5-sonnet.csv")
# reclassifed_themes_df.info()

In [ ]:
#reclassifed_themes_df[(reclassifed_themes_df['theme_1'].isna()) & (reclassifed_themes_df['theme_2'].isna()) & (reclassifed_themes_df['theme_3'].isna())]

In [ ]:
# reclassifed_themes_df_1_review = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/racial_theme_analysis_20260106_221456/theme_classification_20260112_195921/theme_classified_reviews_claude-3.5-sonnet.csv")
# reclassifed_themes_df_1_review.info()

In [ ]:
#reclassifed_themes_df_1_review[(reclassifed_themes_df_1_review['theme_1'].isna()) & (reclassifed_themes_df_1_review['theme_2'].isna()) & (reclassifed_themes_df_1_review['theme_3'].isna())]

In [ ]:
# ##merge the reclassified themes back to the main analysis df
# # List of columns to update (excluding review_id which is the key)
# columns_to_update = [
#     'batch_id', 'theme_1', 'confidence_1', 'theme_2', 
#     'confidence_2', 'theme_3', 'confidence_3', 'all_themes', 
#     'theme_count', 'error'
# ]

# # Combine filling datasets
# df_fill = pd.concat([reclassifed_themes_df, reclassifed_themes_df_1_review], ignore_index=True)
# df_fill = df_fill.dropna(subset=['theme_1'], how='all')  # Ensure there's at least one theme
# df_fill.info()
# # Create a mapping dictionary for each column
# for column in columns_to_update:
#     if column in racial_reviews.columns and column in df_fill.columns:
#         # Create a Series mapping review_id to value
#         mapping = df_fill.set_index('review_id')[column]
#         # Update racial_reviews where values are null
#         mask = racial_reviews[column].isnull() & racial_reviews['review_id'].isin(mapping.index)
#         racial_reviews.loc[mask, column] = racial_reviews.loc[mask, 'review_id'].map(mapping)

# print(f"Null values remaining: {racial_reviews[columns_to_update].isnull().sum().sum()}")

In [ ]:
# # racial_reviews[(racial_reviews['theme_1'].isna()) & (racial_reviews['theme_2'].isna()) & (racial_reviews['theme_3'].isna())]

In [ ]:
## resave the file
#racial_reviews.to_parquet("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/final_merged_data_for_analysis_with_cleaned_classifications_07012026.parquet", engine='fastparquet')

## create table  on prevalence

In [ ]:
analysis_gdf.groupby('social_encounter').agg({'racial_connotations':'value_counts',
'connotation_type':'value_counts'})

In [ ]:

# Assuming racial_reviews is your main dataset
print(f"Total reviews: {len(racial_reviews)}")
print(f"Total pools (unique place_id): {racial_reviews['place_id'].nunique()}")

# Social encounters
social_reviews = analysis_gdf[analysis_gdf['social_encounter'] == 'yes']
print(f"Social encounter reviews: {len(social_reviews)}")
print(f"Percentage: {len(social_reviews)/len(analysis_gdf)*100:.1f}%")

print(f"Number of constituencies within social encounters: {len(social_reviews.constituency.unique())}")
print(f"Number of states with at least one racial connotation review: {len(racial_reviews_gdf.state.unique())}")
print(f"Number of constituencies with at least one racial connotation review: {len(racial_reviews_gdf.constituency.unique())}")
print(f"Percentage of total reviews: {len(racial_reviews_gdf)/len(analysis_gdf)*100:.3f}%")
print(f"Percentage of social reviews: {len(racial_reviews_gdf)/len(social_reviews)*100:.1f}%")

In [ ]:
#fid the difference in the constituencies in the voters demographic data and the constituencies in the social reviews with racial connotations
social_constituencies = set(social_reviews['constituency'].unique())
racial_constituencies = set(racial_reviews_gdf['constituency'].unique())
constituency_difference = social_constituencies.symmetric_difference(racial_constituencies)
print("Constituencies in social reviews but not in racial reviews:", constituency_difference)

In [ ]:
analysis_gdf[analysis_gdf['constituency'].isin(constituency_difference)][['place_id', 'total_reviews', 'state']].drop_duplicates()

In [ ]:
racial_reviews_gdf[~racial_reviews_gdf['profile_pic'].isna()][['review_id', 'profile_pic']]

## reviewer characterization of racial reviews

In [ ]:
# check for 

In [ ]:
racial_reviews_gdf[racial_reviews_gdf['reviewer_id'].isnull()]

In [ ]:
racial_reviews_gdf.reviewer_id.nunique()

In [ ]:
# check for reviewer that have multiple reviews and if it is for the same place_id
racial_reviews_gdf.groupby('reviewer_id').agg({'place_id':'nunique', 'review_id':'count'}).sort_values(by='review_id', ascending=False).head(30)

In [ ]:
racial_reviews[racial_reviews['ethnicity'].isnull()]

In [ ]:
analysis_gdf[analysis_gdf['ethnicity'].isnull()]

In [ ]:
# Count reviewers with more than 1 review
review_counts = racial_reviews_gdf.groupby('reviewer_id')['review_id'].count()
reviewers_with_multiple = (review_counts > 1).sum()
print(f"Number of reviewers with multiple reviews: {reviewers_with_multiple}")
print(f"Total reviewers: {len(review_counts)}")
print(f"Percentage with multiple racialreviews: {reviewers_with_multiple/len(review_counts)*100:.1f}%")

In [ ]:
# check for reviewer that have multiple reviews and if it is for the same place_id
analysis_gdf.groupby('reviewer_id').agg({'place_id':'nunique', 'review_id':'count'}).sort_values(by='review_id', ascending=False).head(30)

In [ ]:
# Count reviewers with more than 1 review
review_counts = analysis_gdf.groupby('reviewer_id')['review_id'].count()
reviewers_with_multiple = (review_counts > 1).sum()
print(f"Number of reviewers with multiple reviews: {reviewers_with_multiple}")
print(f"Total reviewers: {len(review_counts)}")
print(f"Percentage with multiple reviews: {reviewers_with_multiple/len(review_counts)*100:.1f}%")

In [ ]:
# 4. nationality distribution analysis
if 'nationality' in analysis_gdf.columns:
    # Check data type and clean if needed
    if analysis_gdf['nationality'].dtype == 'object':
        # Convert to string and clean
        analysis_gdf['nationality_clean'] = analysis_gdf['nationality'].astype(str).str.strip()
        
        # Replace 'nan' string with actual NaN
        analysis_gdf['nationality_clean'] = analysis_gdf['nationality_clean'].replace(['nan', 'NaN', 'None', ''], np.nan)
    
    # Calculate nationality distribution
    nationality_counts = analysis_gdf['nationality_clean'].value_counts(dropna=False)
    nationality_pct = analysis_gdf['nationality_clean'].value_counts(normalize=True, dropna=False) * 100
    
    
    print("\n=== nationality Distribution of Reviewers Leaving Non racial Reviews ===")
    print(f"Total with nationality data: {analysis_gdf['nationality_clean'].notna().sum()}")
    print(f"Missing nationality: {nationality_pct.get(np.nan, 0):.1f}%")
    print("\nTop ethnicities:")
    # Sort by percentage (excluding NaN)
    sorted_pct = nationality_pct.dropna().sort_values(ascending=False)
    for nationality, pct in sorted_pct.head(10).items():
        print(f"  {nationality}: {pct:.1f}%")

In [ ]:
# 4. nationality distribution analysis
if 'nationality' in racial_reviews_gdf.columns:
    # Check data type and clean if needed
    if racial_reviews_gdf['nationality'].dtype == 'object':
        # Convert to string and clean
        racial_reviews_gdf['nationality_clean'] = racial_reviews_gdf['nationality'].astype(str).str.strip()
        
        # Replace 'nan' string with actual NaN
        racial_reviews_gdf['nationality_clean'] = racial_reviews_gdf['nationality_clean'].replace(['nan', 'NaN', 'None', ''], np.nan)
    
    # Calculate nationality distribution
    nationality_counts = racial_reviews_gdf['nationality_clean'].value_counts(dropna=False)
    nationality_pct = racial_reviews_gdf['nationality_clean'].value_counts(normalize=True, dropna=False) * 100
    
    
    print("\n=== nationality Distribution of Reviewers Leaving Racial Reviews ===")
    print(f"Total with nationality data: {racial_reviews_gdf['nationality_clean'].notna().sum()}")
    print(f"Missing nationality: {nationality_pct.get(np.nan, 0):.1f}%")
    print("\nTop ethnicities:")
    # Sort by percentage (excluding NaN)
    sorted_pct = nationality_pct.dropna().sort_values(ascending=False)
    for nationality, pct in sorted_pct.head(10).items():
        print(f"  {nationality}: {pct:.1f}%")

In [ ]:
# reviewer characterization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Create directory for figures
os.makedirs('figures', exist_ok=True)

# Load data
df = analysis_gdf.copy()

# Filter racial reviews
print(f"Total racial reviews: {len(racial_reviews_gdf)}")
print(f"Unique reviewers of racial reviews: {racial_reviews_gdf['reviewer_id'].nunique()}")

# Calculate reviewer statistics
reviewer_stats = {}

# 1. Number of other reviews (median, mean)
reviewer_stats['mean_other_reviews'] = racial_reviews_gdf['other_reviews'].mean()
reviewer_stats['median_other_reviews'] = racial_reviews_gdf['other_reviews'].median()
reviewer_stats['min_other_reviews'] = racial_reviews_gdf['other_reviews'].min()
reviewer_stats['max_other_reviews'] = racial_reviews_gdf['other_reviews'].max()
reviewer_stats['std_other_reviews'] = racial_reviews_gdf['other_reviews'].std()

#2. number of uploaded pics
reviewer_stats['mean_uploaded_pics'] = racial_reviews_gdf['uploaded_pics'].mean()
reviewer_stats['median_uploaded_pics'] = racial_reviews_gdf['uploaded_pics'].median()
reviewer_stats['min_uploaded_pics'] = racial_reviews_gdf['uploaded_pics'].min()
reviewer_stats['max_uploaded_pics'] = racial_reviews_gdf['uploaded_pics'].max()
reviewer_stats['std_uploaded_pics'] = racial_reviews_gdf['uploaded_pics'].std()

# 2. Profile picture presence
racial_reviews_gdf['has_profile_pic'] = ~racial_reviews_gdf['profile_pic'].isna()
reviewer_stats['pct_with_profile_pic'] = racial_reviews_gdf['has_profile_pic'].mean() * 100
print(f"Percentage with profile picture: {reviewer_stats['pct_with_profile_pic']:.1f}%")
# 3. Reviewer name analysis - UPDATED based on ethnicity
# Check if names appear to be real or pseudonyms
# Using ethnicity as proxy: if ethnicity is missing, likely a pseudonym
def is_pseudonym(row):
    name = row['reviewer_name']
    ethnicity = row['ethnicity']
    
    # If ethnicity is NaN or null, consider it a pseudonym
    if pd.isna(ethnicity):
        return True
    
    # Additional check: if name is single word or contains unusual characters
    name_str = str(name).strip() if not pd.isna(name) else ""
    
    if len(name_str.split()) == 1:
        return True
    if any(char.isdigit() for char in name_str):
        return True
    if any(char in '@#$%^&*()_+=' for char in name_str):
        return True
    
    return False

# Check if ethnicity column exists before using it
if 'ethnicity' in racial_reviews_gdf.columns:
    racial_reviews_gdf['is_pseudonym'] = racial_reviews_gdf.apply(is_pseudonym, axis=1)
    reviewer_stats['pct_pseudonyms'] = racial_reviews_gdf['is_pseudonym'].mean() * 100
else:
    print("Warning: Ethnicity column not found, cannot determine pseudonyms")
    racial_reviews_gdf['is_pseudonym'] = False
    reviewer_stats['pct_pseudonyms'] = None

# 4. Ethnicity distribution analysis
if 'ethnicity' in racial_reviews_gdf.columns:
    # Check data type and clean if needed
    if racial_reviews_gdf['ethnicity'].dtype == 'object':
        # Convert to string and clean
        racial_reviews_gdf['ethnicity_clean'] = racial_reviews_gdf['ethnicity'].astype(str).str.strip()
        
        # Replace 'nan' string with actual NaN
        racial_reviews_gdf['ethnicity_clean'] = racial_reviews_gdf['ethnicity_clean'].replace(['nan', 'NaN', 'None', ''], np.nan)
    
    # Calculate ethnicity distribution
    ethnicity_counts = racial_reviews_gdf['ethnicity_clean'].value_counts(dropna=False)
    ethnicity_pct = racial_reviews_gdf['ethnicity_clean'].value_counts(normalize=True, dropna=False) * 100
    
    reviewer_stats['ethnicity_distribution'] = ethnicity_counts
    reviewer_stats['ethnicity_percentages'] = ethnicity_pct
    
    print("\n=== Ethnicity Distribution of Reviewers Leaving Racial Reviews ===")
    print(f"Total with ethnicity data: {racial_reviews_gdf['ethnicity_clean'].notna().sum()}")
    print(f"Missing ethnicity: {ethnicity_pct.get(np.nan, 0):.1f}%")
    print("\nTop ethnicities:")
    # Sort by percentage (excluding NaN)
    sorted_pct = ethnicity_pct.dropna().sort_values(ascending=False)
    for ethnicity, pct in sorted_pct.head(10).items():
        print(f"  {ethnicity}: {pct:.1f}%")
    
    # Create visualization for ethnicity distribution
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Get top 10 ethnicities (excluding NaN)
    top_ethnicities = sorted_pct.head(10)
    
    # Create bar chart
    bars = ax.bar(range(len(top_ethnicities)), top_ethnicities.values, color='steelblue')
    ax.set_xticks(range(len(top_ethnicities)))
    ax.set_xticklabels(top_ethnicities.index, rotation=45, ha='right')
    ax.set_ylabel('Percentage of Reviewers')
    ax.set_title('Top 10 Ethnicities Among Reviewers of Racial Content')
    
    # Add percentage labels on bars
    for bar, value in zip(bars, top_ethnicities.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'{value:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    fig.savefig('figures/ethnicity_distribution.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    
else:
    print("Ethnicity column not found in data")

# 5. Reviewer activity level
def classify_reviewer_activity(other_reviews):
    if pd.isna(other_reviews):
        return 'Unknown'
    elif other_reviews == 0:
        return 'First-time reviewer'
    elif other_reviews <= 5:
        return 'Occasional reviewer'
    elif other_reviews <= 20:
        return 'Active reviewer'
    else:
        return 'Power reviewer'

racial_reviews_gdf['reviewer_activity'] = racial_reviews_gdf['other_reviews'].apply(classify_reviewer_activity)
activity_counts = racial_reviews_gdf['reviewer_activity'].value_counts()
activity_pct = racial_reviews_gdf['reviewer_activity'].value_counts(normalize=True) * 100

reviewer_stats['activity_distribution'] = activity_counts
reviewer_stats['activity_percentages'] = activity_pct

# 6. Compare with non-racial reviewers for context
non_racial_reviews = df[(df['racial_connotations'] == 'no') | (df['racial_connotations']== 'N/A')].copy()

print(f"Total non racial reviews: {len(non_racial_reviews)}")
print(f"Unique reviewers of non racial reviews: {non_racial_reviews['reviewer_id'].nunique()}")
# Add same features to non_racial_reviews for comparison
non_racial_reviews['has_profile_pic'] = ~non_racial_reviews['profile_pic'].isna()
print(f"Percentage with profile picture (non-racial reviewers): {non_racial_reviews['has_profile_pic'].mean() * 100:.1f}%") 



if 'ethnicity' in non_racial_reviews.columns:
    non_racial_reviews['is_pseudonym'] = non_racial_reviews.apply(is_pseudonym, axis=1)
else:
    non_racial_reviews['is_pseudonym'] = False


comparison_stats = pd.DataFrame({
    'Racial_Reviewers': [
        racial_reviews_gdf['other_reviews'].mean(),
        racial_reviews_gdf['other_reviews'].median(),
        racial_reviews_gdf['other_reviews'].min(),
        racial_reviews_gdf['other_reviews'].max(),
        racial_reviews_gdf['other_reviews'].std(),
        racial_reviews_gdf['has_profile_pic'].mean() * 100,
        racial_reviews_gdf['is_pseudonym'].mean() * 100 if 'is_pseudonym' in racial_reviews_gdf.columns else 0
    ],
    'Non_Racial_Reviewers': [
        non_racial_reviews['other_reviews'].mean(),
        non_racial_reviews['other_reviews'].median(),
        non_racial_reviews['other_reviews'].min(),
        non_racial_reviews['other_reviews'].max(),
        non_racial_reviews['other_reviews'].std(),
        non_racial_reviews['has_profile_pic'].mean() * 100,
        non_racial_reviews['is_pseudonym'].mean() * 100 if 'is_pseudonym' in non_racial_reviews.columns else 0
    ]
}, index=['Mean Other Reviews', 'Median Other Reviews', 'Min Other Reviews', 
          'Max Other Reviews', 'Std Other Reviews', 
          '% With Profile Picture', '% Using Pseudonyms'])

print("\n=== Comparison Statistics ===")
print(comparison_stats)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Other reviews distribution
axes[0, 0].hist(racial_reviews_gdf['other_reviews'].dropna(), bins=50, alpha=0.7, color='steelblue')
axes[0, 0].axvline(racial_reviews_gdf['other_reviews'].median(), color='red', linestyle='--', 
                   label=f'Median: {racial_reviews_gdf["other_reviews"].median():.1f}')
axes[0, 0].axvline(racial_reviews_gdf['other_reviews'].mean(), color='green', linestyle='--', 
                   label=f'Mean: {racial_reviews_gdf["other_reviews"].mean():.1f}')
axes[0, 0].set_xlabel('Number of Other Reviews')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Distribution of Other Reviews for Reviewers of Racial Content')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 100)  # Cap for visualization

# Plot 2: Profile picture presence
profile_data = [racial_reviews_gdf['has_profile_pic'].mean() * 100, 
                non_racial_reviews['has_profile_pic'].mean() * 100]
bars1 = axes[0, 1].bar(['Racial Reviewers', 'Non-Racial Reviewers'], profile_data, 
                       color=['steelblue', 'lightgray'])
axes[0, 1].set_ylabel('Percentage with Profile Picture')
axes[0, 1].set_title('Profile Picture Presence')
axes[0, 1].set_ylim(0, 100)
for bar, value in zip(bars1, profile_data):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                   f'{value:.1f}%', ha='center', va='bottom')

# Plot 3: Pseudonym usage (if available)
if 'is_pseudonym' in racial_reviews_gdf.columns:
    pseudonym_data = [racial_reviews_gdf['is_pseudonym'].mean() * 100, 
                      non_racial_reviews['is_pseudonym'].mean() * 100]
    bars2 = axes[1, 0].bar(['Racial Reviewers', 'Non-Racial Reviewers'], pseudonym_data,
                           color=['steelblue', 'lightgray'])
    axes[1, 0].set_ylabel('Percentage Using Pseudonyms')
    axes[1, 0].set_title('Pseudonym Usage (Based on Missing Ethnicity)')
    axes[1, 0].set_ylim(0, 100)
    for bar, value in zip(bars2, pseudonym_data):
        axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                       f'{value:.1f}%', ha='center', va='bottom')
else:
    axes[1, 0].text(0.5, 0.5, 'Pseudonym data not available\n(Ethnicity column missing)', 
                    ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('Pseudonym Usage')

# Plot 4: Reviewer activity level
activity_data = activity_pct.sort_values(ascending=False)
bars3 = axes[1, 1].barh(range(len(activity_data)), activity_data.values, color='steelblue')
axes[1, 1].set_yticks(range(len(activity_data)))
axes[1, 1].set_yticklabels(activity_data.index)
axes[1, 1].set_xlabel('Percentage of Reviewers')
axes[1, 1].set_title('Reviewer Activity Level')
for i, (bar, value) in enumerate(zip(bars3, activity_data.values)):
    axes[1, 1].text(bar.get_width() + 1, i, f'{value:.1f}%', va='center')

plt.tight_layout()
fig.savefig('figures/reviewer_characteristics.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print summary table
print("\n=== REVIEWER CHARACTERISTICS SUMMARY ===")
print(f"Total reviewers of racial content: {racial_reviews_gdf['reviewer_id'].nunique()}")
print(f"Mean number of other reviews: {reviewer_stats['mean_other_reviews']:.2f}")
print(f"Median number of other reviews: {reviewer_stats['median_other_reviews']:.2f}")
print(f"Percentage with profile picture: {reviewer_stats['pct_with_profile_pic']:.1f}%")

if reviewer_stats['pct_pseudonyms'] is not None:
    print(f"Percentage using pseudonyms: {reviewer_stats['pct_pseudonyms']:.1f}%")
else:
    print("Percentage using pseudonyms: Not available (ethnicity column missing)")

print("\nReviewer Activity Level:")
for activity, pct in activity_pct.items():
    print(f"  {activity}: {pct:.1f}%")

## significance of racial reviews

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')
# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("colorblind")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

In [ ]:
def aggregate_constituency_data(analysis_gdf, demo_df=None):
        """
        Aggregate review data to constituency level and merge with demographic data.
        """
        # Group reviews by constituency
        analysis_gdf['is_racial'] = analysis_gdf['racial_connotations'].apply(lambda x: 1 if x == 'yes' else 0) 
        constituency_stats = analysis_gdf.groupby('constituency_nr').agg(
            total_reviews=('review_id', 'count'),
            racial_reviews=('is_racial', 'sum'),
            discriminatory_reviews=('connotation_type', lambda x: (x == 'discriminatory').sum()),
            ambiguous_reviews=('connotation_type', lambda x: (x == 'ambiguous').sum()),
            pleasant_reviews=('connotation_type', lambda x: (x == 'pleasant').sum()),
            pool_count=('place_id', pd.Series.nunique)
        ).reset_index()
        
        # Calculate proportions
        constituency_stats['racial_proportion'] = (constituency_stats['racial_reviews'] / 
                                                  constituency_stats['total_reviews'].replace(0, np.nan)) * 100
        constituency_stats['racial_per_pool'] = constituency_stats['racial_reviews'] / constituency_stats['pool_count'].replace(0, np.nan)
        constituency_stats['racial_per_1000_reviews'] = (constituency_stats['racial_reviews'] / 
                                                        constituency_stats['total_reviews'].replace(0, np.nan)) * 1000
        
        # Calculate connotation proportions among racial reviews
        constituency_stats['discriminatory_prop'] = (constituency_stats['discriminatory_reviews'] / 
                                                    constituency_stats['racial_reviews'].replace(0, np.nan)) * 100
        constituency_stats['pleasant_prop'] = (constituency_stats['pleasant_reviews'] / 
                                              constituency_stats['racial_reviews'].replace(0, np.nan)) * 100
        
        # Merge with demographic data if available
        if demo_df is not None:
            constituency_df = pd.merge(constituency_stats, demo_df, 
                                           on='constituency_nr', how='left')
            print(f"Merged with demographic data for {len(constituency_df)} constituencies")
        else:
            constituency_df = constituency_stats
            print(f"Aggregated data for {len(constituency_df)} constituencies (no demographic data)")
            
    
        return constituency_df
constituency_df = aggregate_constituency_data(analysis_gdf, demographic_voters_data)
constituency_df.info()

In [ ]:
affected_constituencies = constituency_df[constituency_df['racial_reviews'] > 0].copy()
if len(affected_constituencies) == 0:
    print("Warning: No constituencies with racial reviews found!")
    # Create empty summary
    summary_stats = pd.DataFrame({
    'Metric': ['No racial reviews found'],
    'Value': ['0']
    })


# Initialize correlation variables to NaN
foreign_pop_corr = np.nan
afd_corr = np.nan

if 'population_foreign_percent' in affected_constituencies.columns:
    valid_data = affected_constituencies[['racial_proportion', 'population_foreign_percent']].dropna()
    if len(valid_data) > 1:
        foreign_pop_corr = valid_data['racial_proportion'].corr(valid_data['population_foreign_percent'])

if 'afd_vote_share' in affected_constituencies.columns:
    valid_data = affected_constituencies[['racial_proportion', 'afd_vote_share']].dropna()
    if len(valid_data) > 1:
        afd_corr = valid_data['racial_proportion'].corr(valid_data['afd_vote_share'])

# Create summary statistics
summary_data = {
    'Number of constituencies with racial discourse': len(affected_constituencies),
    'Total constituencies analyzed': len(constituency_df),
    'Mean racial reviews per constituency': affected_constituencies['racial_reviews'].mean(),
    'Median racial reviews per constituency': affected_constituencies['racial_reviews'].median(),
    'Std dev of racial reviews': affected_constituencies['racial_reviews'].std(),
    'Max racial reviews in a constituency': affected_constituencies['racial_reviews'].max(),
    'Min racial reviews in a constituency': affected_constituencies['racial_reviews'].min(),
    'Mean racial proportion (%)': affected_constituencies['racial_proportion'].mean(),
    'Median racial proportion (%)': affected_constituencies['racial_proportion'].median(),
    'Mean racial per 1000 reviews': affected_constituencies['racial_per_1000_reviews'].mean(),
}

# Add demographic statistics if available
if 'population_foreign_percent' in affected_constituencies.columns:
    summary_data['Mean foreign population (%)'] = affected_constituencies['population_foreign_percent'].mean()

if 'afd_vote_share' in affected_constituencies.columns:
    summary_data['Mean AfD vote share (%)'] = affected_constituencies['afd_vote_share'].mean()

if not np.isnan(foreign_pop_corr):
    summary_data['Correlation (racial prop. vs foreign pop.)'] = foreign_pop_corr

if not np.isnan(afd_corr):
    summary_data['Correlation (racial prop. vs AfD share)'] = afd_corr

# Create DataFrame
summary_stats = pd.DataFrame({
    'Metric': list(summary_data.keys()),
    'Value': list(summary_data.values())
})

# Format values
def format_value(x):
    if isinstance(x, (int, np.integer)):
        return str(x)
    elif isinstance(x, float):
        if np.isnan(x):
            return 'N/A'
        return f"{x:.3f}"
    return str(x)

summary_stats['Value'] = summary_stats['Value'].apply(format_value)
display(summary_stats)

In [ ]:
    
def identify_outliers(constituency_df, n=10):
        """
        Identify top constituencies by different metrics.
        """
        if constituency_df is None:
            aggregate_constituency_data()
            
        # Filter only constituencies with racial reviews
        affected_constituencies = constituency_df[constituency_df['racial_reviews'] > 0].copy()
        
        if len(affected_constituencies) == 0:
            print("No outliers identified: No constituencies with racial reviews")
            return {}
        
        outliers = {
            'top_racial_count': affected_constituencies.nlargest(min(n, len(affected_constituencies)), 'racial_reviews'),
            'top_racial_proportion': affected_constituencies.nlargest(min(n, len(affected_constituencies)), 'racial_proportion')
        }
        
        # Add demographic-based outliers if available
        if 'afd_vote_share' in affected_constituencies.columns and len(affected_constituencies) >= 4:
            outliers['top_afd_high_discourse'] = affected_constituencies.nlargest(n, 'afd_vote_share')
            
            q75_racial = affected_constituencies['racial_proportion'].quantile(0.75)
            q25_afd = affected_constituencies['afd_vote_share'].quantile(0.25)
            high_discourse_low_afd = affected_constituencies[
                (affected_constituencies['racial_proportion'] > q75_racial) &
                (affected_constituencies['afd_vote_share'] < q25_afd)
            ]
            outliers['high_discourse_low_afd'] = high_discourse_low_afd
        
        if 'population_foreign_percent' in affected_constituencies.columns and len(affected_constituencies) >= 4:
            outliers['top_foreignpop_high_discourse'] = affected_constituencies.nlargest(n, 'population_foreign_percent')
            
            q25_racial = affected_constituencies['racial_proportion'].quantile(0.25)
            q75_foreign = affected_constituencies['population_foreign_percent'].quantile(0.75)
            low_discourse_high_foreignpop = affected_constituencies[
                (affected_constituencies['racial_proportion'] < q25_racial) &
                (affected_constituencies['population_foreign_percent'] > q75_foreign)
            ]
            outliers['low_discourse_high_foreignpop'] = low_discourse_high_foreignpop
        
        return outliers
outliers = identify_outliers(constituency_df, n=10)
for key, df_outlier in outliers.items():
    print(f"\n=== Outlier Category: {key} ===")
    display(df_outlier[['constituency_nr', 'constituency', 'state', 'racial_reviews', 'racial_proportion']])
   

In [ ]:
def run_negative_binomial_regression(constituency_df):
    """
    Run negative binomial regression on constituency data.
    
    Parameters:
    -----------
    constituency_df : pandas.DataFrame
        DataFrame containing constituency data with required columns
        
    Returns:
    --------
    model : statsmodels model object or None
    results_df : DataFrame with regression results
    diagnostics : dict with model diagnostics
    """
    import pandas as pd
    import numpy as np
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    
    if constituency_df is None or constituency_df.empty:
        print("Constituency data is None or empty")
        return None, pd.DataFrame(), {}
    
    # Check if we have required columns
    required_cols = ['racial_reviews', 'total_reviews']
    missing_req = [col for col in required_cols if col not in constituency_df.columns]
    if missing_req:
        print(f"Missing required columns: {missing_req}")
        return None, pd.DataFrame(), {}
    
    # Define all possible demographic variables
    possible_demo_cols = ['population_foreign_percent', 'afd', 'urbanization_index']
    available_demo_cols = [col for col in possible_demo_cols 
                          if col in constituency_df.columns]
    
    if len(available_demo_cols) == 0:
        print("No demographic variables available for regression")
        return None, pd.DataFrame(), {}
    
    # Prepare data for regression
    reg_data = constituency_df[required_cols + available_demo_cols].copy()
    reg_data = reg_data.dropna()
    
    # Check for sufficient sample size
    min_samples = max(10, 2 * len(available_demo_cols) + 5)  # Rule of thumb
    if len(reg_data) < min_samples:
        print(f"Insufficient data for regression: {len(reg_data)} complete cases, "
              f"need at least {min_samples}")
        return None, pd.DataFrame(), {}
    
    # Add offset (log of total reviews) - handle zeros
    reg_data['log_total_reviews'] = np.log(reg_data['total_reviews'].replace(0, 0.5))
    
    # Define formula
    formula = f'racial_reviews ~ {" + ".join(available_demo_cols)}'
    
    try:
        # Fit Negative Binomial model
        model = smf.glm(formula=formula,
                       data=reg_data,
                       family=sm.families.NegativeBinomial(),
                       offset=reg_data['log_total_reviews']).fit(maxiter=100)
        
        # Create comprehensive results DataFrame
        results_df = pd.DataFrame({
            'coef': model.params,
            'std_err': model.bse,
            'z_value': model.tvalues,
            'p_value': model.pvalues,
            'ci_lower': model.conf_int()[0],
            'ci_upper': model.conf_int()[1],
            'IRR': np.exp(model.params),
            'IRR_ci_lower': np.exp(model.conf_int()[0]),
            'IRR_ci_upper': np.exp(model.conf_int()[1])
        })
        
        # Add significance stars
        results_df['significance'] = results_df['p_value'].apply(
            lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        )
        
        # Enhanced diagnostics
        diagnostics = {
            'AIC': model.aic,
            'BIC': model.bic,
            'Log-likelihood': model.llf,
            'Deviance': model.deviance,
            'Null_deviance': model.null_deviance,
            'Pearson_chi2': model.pearson_chi2,
            'df_resid': model.df_resid,
            'df_model': model.df_model,
            'n_obs': model.nobs,
            'converged': model.converged,
            'iteration_count': getattr(model, 'iteration', None),
            'pseudo_r2': 1 - (model.deviance / model.null_deviance),
            'dispersion': model.scale,
            'formula': formula,
            'predictors': available_demo_cols,
            'offset_used': True
        }
        
        # Check for convergence issues
        if not model.converged:
            print("Warning: Model did not converge properly")
        
        # Print summary
        print(f"Negative binomial regression fitted successfully")
        print(f"n = {model.nobs}, AIC = {model.aic:.1f}, BIC = {model.bic:.1f}")
        print(f"Pseudo R² = {diagnostics['pseudo_r2']:.3f}")
        print(f"Predictors: {', '.join(available_demo_cols)}")
        
        return model, results_df, diagnostics
        
    except Exception as e:
        print(f"Error fitting negative binomial model: {e}")
        import traceback
        traceback.print_exc()
        return None, pd.DataFrame(), {}
    

run_negative_binomial_regression(constituency_df)
model, results_df, diagnostics = run_negative_binomial_regression(constituency_df)  
results_df


In [ ]:
def plot_scatter_relationships(, save_path=None):
        """
        Create scatter plots of racial proportion vs demographic variables.
        """
        if .constituency_df is None:
            .aggregate_constituency_data()
            
        affected_constituencies = .constituency_df[.constituency_df['racial_reviews'] > 0]
        
        if len(affected_constituencies) == 0:
            print("No constituencies with racial reviews to plot")
            return None, None
        
        # Determine which demographic variables are available
        plots_to_create = []
        if 'population_foreign_percent' in affected_constituencies.columns:
            plots_to_create.append('foreign_pop')
        if 'afd_vote_share' in affected_constituencies.columns:
            plots_to_create.append('afd_share')
        
        if len(plots_to_create) == 0:
            print("No demographic data available for scatter plots")
            return None, None
        
        fig, axes = plt.subplots(1, len(plots_to_create), figsize=(6 * len(plots_to_create), 5))
        
        if len(plots_to_create) == 1:
            axes = [axes]
        
        plot_index = 0
        
        # Plot 1: Racial proportion vs Foreign population %
        if 'population_foreign_percent' in affected_constituencies.columns:
            ax = axes[plot_index]
            plot_index += 1
            
            # Choose color variable
            if 'afd_vote_share' in affected_constituencies.columns:
                colors = affected_constituencies['afd_vote_share']
                cmap = 'RdBu_r'
            else:
                colors = 'blue'
                cmap = None
            
            sc = ax.scatter(affected_constituencies['population_foreign_percent'], 
                           affected_constituencies['racial_proportion'],
                           c=colors, cmap=cmap, alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
            
            # Add regression line
            valid_data = affected_constituencies[['population_foreign_percent', 'racial_proportion']].dropna()
            if len(valid_data) > 1:
                z = np.polyfit(valid_data['population_foreign_percent'], 
                              valid_data['racial_proportion'], 1)
                p = np.poly1d(z)
                x_range = np.linspace(valid_data['population_foreign_percentt'].min(), 
                                     valid_data['population_foreign_percentt'].max(), 100)
                ax.plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
            
            ax.set_xlabel('Foreign Population (%)', fontsize=11)
            ax.set_ylabel('Racialized Reviews (%)', fontsize=11)
            ax.set_title('Racial Discourse vs Foreign Population', fontsize=12, fontweight='bold')
            
            # Calculate correlation
            corr = valid_data['racial_proportion'].corr(valid_data['population_foreign_percentt'])
            if not np.isnan(corr):
                ax.text(0.05, 0.95, f'ρ = {corr:.2f}', transform=ax.transAxes, 
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # Add colorbar if using AfD vote share for coloring
            if 'afd_vote_share' in affected_constituencies.columns:
                cbar = plt.colorbar(sc, ax=ax)
                cbar.set_label('AfD Vote Share (%)', fontsize=10)
        
        # Plot 2: Racial proportion vs AfD vote share
        if 'afd_vote_share' in affected_constituencies.columns:
            ax = axes[plot_index] if plot_index < len(axes) else axes[0]
            
            # Choose color variable
            if 'population_foreign_percentt' in affected_constituencies.columns:
                colors = affected_constituencies['population_foreign_percentt']
                cmap = 'viridis'
            else:
                colors = 'blue'
                cmap = None
            
            sc = ax.scatter(affected_constituencies['afd_vote_share'], 
                           affected_constituencies['racial_proportion'],
                           c=colors, cmap=cmap, alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
            
            # Add regression line
            valid_data = affected_constituencies[['afd_vote_share', 'racial_proportion']].dropna()
            if len(valid_data) > 1:
                z = np.polyfit(valid_data['afd_vote_share'], 
                              valid_data['racial_proportion'], 1)
                p = np.poly1d(z)
                x_range = np.linspace(valid_data['afd_vote_share'].min(), 
                                     valid_data['afd_vote_share'].max(), 100)
                ax.plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
            
            ax.set_xlabel('AfD Vote Share (%)', fontsize=11)
            ax.set_ylabel('Racialized Reviews (%)', fontsize=11)
            ax.set_title('Racial Discourse vs AfD Support', fontsize=12, fontweight='bold')
            
            # Calculate correlation
            corr = valid_data['racial_proportion'].corr(valid_data['afd_vote_share'])
            if not np.isnan(corr):
                ax.text(0.05, 0.95, f'ρ = {corr:.2f}', transform=ax.transAxes, 
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # Add colorbar if using foreign population for coloring
            if 'population_foreign_percentt' in affected_constituencies.columns:
                cbar = plt.colorbar(sc, ax=ax)
                cbar.set_label('Foreign Population (%)', fontsize=10)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to {save_path}")
        
        plt.show()
        
        return fig, axes
    

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')

# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("colorblind")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

class RacialDiscourseAnalyzer:
    """
    Analyzer for racialized discourse in swimming pool reviews with demographic context.
    """
    
    def __init__(self, reviews_df, shapefile_path):
        """
        Initialize the analyzer with reviews data and shapefile.
        """
        # Make a copy to avoid modifying original data
        self.reviews_df = reviews_df.copy()
        self.shapefile = gpd.read_file(shapefile_path)
        
        # Ensure constituency_nr is string type
        self.reviews_df['constituency_nr'] = self.reviews_df['constituency_nr'].astype(str)
        
        # Initialize attributes
        self.demo_df = None
        self.constituency_df = None
        self.regression_results = None
        
    def load_demographic_data(self, demo_file_path):
        """
        Load demographic and political data for constituencies.
        """
        self.demo_df = pd.read_csv(demo_file_path)
        self.demo_df['constituency_nr'] = self.demo_df['constituency_nr'].astype(str)
        print(f"Loaded demographic data for {len(self.demo_df)} constituencies")
        return self.demo_df
        
    def aggregate_constituency_data(self):
        """
        Aggregate review data to constituency level and merge with demographic data.
        """
        # Group reviews by constituency
        constituency_stats = self.reviews_df.groupby('constituency_nr').agg(
            total_reviews=('review_id', 'count'),
            racial_reviews=('is_racial', 'sum'),
            discriminatory_reviews=('connotation_type', lambda x: (x == 'discriminatory').sum()),
            ambiguous_reviews=('connotation_type', lambda x: (x == 'ambiguous').sum()),
            pleasant_reviews=('connotation_type', lambda x: (x == 'pleasant').sum()),
            pool_count=('place_id', pd.Series.nunique)
        ).reset_index()
        
        # Calculate proportions
        constituency_stats['racial_proportion'] = (constituency_stats['racial_reviews'] / 
                                                  constituency_stats['total_reviews'].replace(0, np.nan)) * 100
        constituency_stats['racial_per_pool'] = constituency_stats['racial_reviews'] / constituency_stats['pool_count'].replace(0, np.nan)
        constituency_stats['racial_per_1000_reviews'] = (constituency_stats['racial_reviews'] / 
                                                        constituency_stats['total_reviews'].replace(0, np.nan)) * 1000
        
        # Calculate connotation proportions among racial reviews
        constituency_stats['discriminatory_prop'] = (constituency_stats['discriminatory_reviews'] / 
                                                    constituency_stats['racial_reviews'].replace(0, np.nan)) * 100
        constituency_stats['pleasant_prop'] = (constituency_stats['pleasant_reviews'] / 
                                              constituency_stats['racial_reviews'].replace(0, np.nan)) * 100
        
        # Merge with demographic data if available
        if self.demo_df is not None:
            self.constituency_df = pd.merge(constituency_stats, self.demo_df, 
                                           on='constituency_nr', how='left')
            print(f"Merged with demographic data for {len(self.constituency_df)} constituencies")
        else:
            self.constituency_df = constituency_stats
            print(f"Aggregated data for {len(self.constituency_df)} constituencies (no demographic data)")
            
        # Add state information from constituency number
        self.constituency_df['state'] = self.constituency_df['constituency_nr'].str[:2].astype(str)
        
        return self.constituency_df
    
    def create_summary_table(self):
        """
        Create a summary statistics table for constituencies.
        """
        if self.constituency_df is None:
            self.aggregate_constituency_data()
            
        # Filter only constituencies with racial reviews
        affected_constituencies = self.constituency_df[self.constituency_df['racial_reviews'] > 0].copy()
        
        if len(affected_constituencies) == 0:
            print("Warning: No constituencies with racial reviews found!")
            # Create empty summary
            summary_stats = pd.DataFrame({
                'Metric': ['No racial reviews found'],
                'Value': ['0']
            })
            return summary_stats, affected_constituencies
        
        # Calculate correlations if columns exist
        foreign_pop_corr = np.nan
        afd_corr = np.nan
        
        if 'population_foreign_percent' in affected_constituencies.columns:
            valid_data = affected_constituencies[['racial_proportion', 'population_foreign_percent']].dropna()
            if len(valid_data) > 1:
                foreign_pop_corr = valid_data['racial_proportion'].corr(valid_data['population_foreign_percent'])
        
        if 'afd_vote_share' in affected_constituencies.columns:
            valid_data = affected_constituencies[['racial_proportion', 'afd_vote_share']].dropna()
            if len(valid_data) > 1:
                afd_corr = valid_data['racial_proportion'].corr(valid_data['afd_vote_share'])
        
        # Create summary statistics
        summary_data = {
            'Number of constituencies with racial discourse': len(affected_constituencies),
            'Total constituencies analyzed': len(self.constituency_df),
            'Mean racial reviews per constituency': affected_constituencies['racial_reviews'].mean(),
            'Median racial reviews per constituency': affected_constituencies['racial_reviews'].median(),
            'Std dev of racial reviews': affected_constituencies['racial_reviews'].std(),
            'Max racial reviews in a constituency': affected_constituencies['racial_reviews'].max(),
            'Mean racial proportion (%)': affected_constituencies['racial_proportion'].mean(),
            'Median racial proportion (%)': affected_constituencies['racial_proportion'].median(),
            'Mean racial per 1000 reviews': affected_constituencies['racial_per_1000_reviews'].mean(),
        }
        
        # Add demographic statistics if available
        if 'population_foreign_percent' in affected_constituencies.columns:
            summary_data['Mean foreign population (%)'] = affected_constituencies['population_foreign_percent'].mean()
        
        if 'afd_vote_share' in affected_constituencies.columns:
            summary_data['Mean AfD vote share (%)'] = affected_constituencies['afd_vote_share'].mean()
        
        if not np.isnan(foreign_pop_corr):
            summary_data['Correlation (racial prop. vs foreign pop.)'] = foreign_pop_corr
        
        if not np.isnan(afd_corr):
            summary_data['Correlation (racial prop. vs AfD share)'] = afd_corr
        
        # Create DataFrame
        summary_stats = pd.DataFrame({
            'Metric': list(summary_data.keys()),
            'Value': list(summary_data.values())
        })
        
        # Format values
        def format_value(x):
            if isinstance(x, (int, np.integer)):
                return str(x)
            elif isinstance(x, float):
                if np.isnan(x):
                    return 'N/A'
                return f"{x:.3f}"
            return str(x)
        
        summary_stats['Value'] = summary_stats['Value'].apply(format_value)
        
        return summary_stats, affected_constituencies
    
    def identify_outliers(self, n=10):
        """
        Identify top constituencies by different metrics.
        """
        if self.constituency_df is None:
            self.aggregate_constituency_data()
            
        # Filter only constituencies with racial reviews
        affected_constituencies = self.constituency_df[self.constituency_df['racial_reviews'] > 0].copy()
        
        if len(affected_constituencies) == 0:
            print("No outliers identified: No constituencies with racial reviews")
            return {}
        
        outliers = {
            'top_racial_count': affected_constituencies.nlargest(min(n, len(affected_constituencies)), 'racial_reviews'),
            'top_racial_proportion': affected_constituencies.nlargest(min(n, len(affected_constituencies)), 'racial_proportion')
        }
        
        # Add demographic-based outliers if available
        if 'afd_vote_share' in affected_constituencies.columns and len(affected_constituencies) >= 4:
            outliers['top_afd_high_discourse'] = affected_constituencies.nlargest(n, 'afd_vote_share')
            
            q75_racial = affected_constituencies['racial_proportion'].quantile(0.75)
            q25_afd = affected_constituencies['afd_vote_share'].quantile(0.25)
            high_discourse_low_afd = affected_constituencies[
                (affected_constituencies['racial_proportion'] > q75_racial) &
                (affected_constituencies['afd_vote_share'] < q25_afd)
            ]
            outliers['high_discourse_low_afd'] = high_discourse_low_afd
        
        if 'population_foreign_percent' in affected_constituencies.columns and len(affected_constituencies) >= 4:
            outliers['top_foreignpop_high_discourse'] = affected_constituencies.nlargest(n, 'population_foreign_percent')
            
            q25_racial = affected_constituencies['racial_proportion'].quantile(0.25)
            q75_foreign = affected_constituencies['population_foreign_percent'].quantile(0.75)
            low_discourse_high_foreignpop = affected_constituencies[
                (affected_constituencies['racial_proportion'] < q25_racial) &
                (affected_constituencies['population_foreign_percent'] > q75_foreign)
            ]
            outliers['low_discourse_high_foreignpop'] = low_discourse_high_foreignpop
        
        return outliers
    
    def run_negative_binomial_regression(self):
        """
        Run negative binomial regression on constituency data.
        
        Returns:
        --------
        model : statsmodels model object
        results_df : DataFrame with regression results
        diagnostics : dict with model diagnostics
        """
        if self.constituency_df is None:
            self.aggregate_constituency_data()
        
        # Check if we have required columns
        required_cols = ['racial_reviews', 'total_reviews']
        available_demo_cols = []
        
        if 'population_foreign_percent' in self.constituency_df.columns:
            available_demo_cols.append('population_foreign_percent')
        if 'afd_vote_share' in self.constituency_df.columns:
            available_demo_cols.append('afd_vote_share')
        if 'urbanization_index' in self.constituency_df.columns:
            available_demo_cols.append('urbanization_index')
        
        if len(available_demo_cols) == 0:
            print("No demographic variables available for regression")
            return None, pd.DataFrame(), {}
        
        # Prepare data for regression
        reg_data = self.constituency_df[['racial_reviews', 'total_reviews'] + available_demo_cols].copy()
        reg_data = reg_data.dropna()
        
        if len(reg_data) < 10:
            print(f"Insufficient data for regression: only {len(reg_data)} complete cases")
            return None, pd.DataFrame(), {}
        
        # Add offset (log of total reviews)
        reg_data['log_total_reviews'] = np.log(reg_data['total_reviews'])
        
        # Define formula
        formula = f'racial_reviews ~ {" + ".join(available_demo_cols)}'
        
        try:
            # Fit Negative Binomial model
            model = smf.glm(formula=formula,
                           data=reg_data,
                           family=sm.families.NegativeBinomial(),
                           offset=reg_data['log_total_reviews']).fit()
            
            # Create results DataFrame
            results_df = pd.DataFrame({
                'coef': model.params,
                'std_err': model.bse,
                'z': model.tvalues,
                'p-value': model.pvalues,
                'IRR': np.exp(model.params)  # Incidence Rate Ratio
            })
            
            # Diagnostics
            diagnostics = {
                'AIC': model.aic,
                'BIC': model.bic,
                'Log-likelihood': model.llf,
                'Deviance': model.deviance,
                'Pearson_chi2': model.pearson_chi2,
                'df_resid': model.df_resid,
                'n_obs': model.nobs
            }
            
            print(f"Negative binomial regression fitted successfully")
            print(f"n = {model.nobs}, AIC = {model.aic:.1f}, BIC = {model.bic:.1f}")
            
            self.regression_results = results_df
            
            return model, results_df, diagnostics
            
        except Exception as e:
            print(f"Error fitting negative binomial model: {e}")
            return None, pd.DataFrame(), {}
    
    def create_publication_table(self):
        """
        Create a publication-ready table of key statistics.
        """
        if self.constituency_df is None:
            self.aggregate_constituency_data()
        
        # Create summary statistics
        summary_stats, affected_constituencies = self.create_summary_table()
        
        # Create publication table
        pub_table = pd.DataFrame({
            'Statistic': [
                'Constituencies analyzed',
                'Constituencies with racial discourse',
                'Racial reviews (total)',
                'Racial reviews per constituency (mean)',
                'Racial proportion (%) (mean)',
                'Racial proportion (%) (median)'
            ],
            'Value': [
                len(self.constituency_df),
                len(affected_constituencies),
                self.constituency_df['racial_reviews'].sum() if len(affected_constituencies) > 0 else 0,
                affected_constituencies['racial_reviews'].mean() if len(affected_constituencies) > 0 else 0,
                affected_constituencies['racial_proportion'].mean() if len(affected_constituencies) > 0 else 0,
                affected_constituencies['racial_proportion'].median() if len(affected_constituencies) > 0 else 0
            ]
        })
        
        # Add demographic statistics if available
        if 'population_foreign_percent' in self.constituency_df.columns and len(affected_constituencies) > 0:
            pub_table = pd.concat([
                pub_table,
                pd.DataFrame({
                    'Statistic': ['Foreign population (%) (mean)', 'AfD vote share (%) (mean)'],
                    'Value': [
                        affected_constituencies['population_foreign_percent'].mean(),
                        affected_constituencies['afd_vote_share'].mean() if 'afd_vote_share' in affected_constituencies.columns else 'N/A'
                    ]
                })
            ], ignore_index=True)
        
        return pub_table
    
    def plot_scatter_relationships(self, save_path=None):
        """
        Create scatter plots of racial proportion vs demographic variables.
        """
        if self.constituency_df is None:
            self.aggregate_constituency_data()
            
        affected_constituencies = self.constituency_df[self.constituency_df['racial_reviews'] > 0]
        
        if len(affected_constituencies) == 0:
            print("No constituencies with racial reviews to plot")
            return None, None
        
        # Determine which demographic variables are available
        plots_to_create = []
        if 'population_foreign_percent' in affected_constituencies.columns:
            plots_to_create.append('foreign_pop')
        if 'afd_vote_share' in affected_constituencies.columns:
            plots_to_create.append('afd_share')
        
        if len(plots_to_create) == 0:
            print("No demographic data available for scatter plots")
            return None, None
        
        fig, axes = plt.subplots(1, len(plots_to_create), figsize=(6 * len(plots_to_create), 5))
        
        if len(plots_to_create) == 1:
            axes = [axes]
        
        plot_index = 0
        
        # Plot 1: Racial proportion vs Foreign population %
        if 'population_foreign_percent' in affected_constituencies.columns:
            ax = axes[plot_index]
            plot_index += 1
            
            # Choose color variable
            if 'afd_vote_share' in affected_constituencies.columns:
                colors = affected_constituencies['afd_vote_share']
                cmap = 'RdBu_r'
            else:
                colors = 'blue'
                cmap = None
            
            sc = ax.scatter(affected_constituencies['population_foreign_percent'], 
                           affected_constituencies['racial_proportion'],
                           c=colors, cmap=cmap, alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
            
            # Add regression line
            valid_data = affected_constituencies[['population_foreign_percent', 'racial_proportion']].dropna()
            if len(valid_data) > 1:
                z = np.polyfit(valid_data['population_foreign_percent'], 
                              valid_data['racial_proportion'], 1)
                p = np.poly1d(z)
                x_range = np.linspace(valid_data['population_foreign_percentt'].min(), 
                                     valid_data['population_foreign_percentt'].max(), 100)
                ax.plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
            
            ax.set_xlabel('Foreign Population (%)', fontsize=11)
            ax.set_ylabel('Racialized Reviews (%)', fontsize=11)
            ax.set_title('Racial Discourse vs Foreign Population', fontsize=12, fontweight='bold')
            
            # Calculate correlation
            corr = valid_data['racial_proportion'].corr(valid_data['population_foreign_percentt'])
            if not np.isnan(corr):
                ax.text(0.05, 0.95, f'ρ = {corr:.2f}', transform=ax.transAxes, 
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # Add colorbar if using AfD vote share for coloring
            if 'afd_vote_share' in affected_constituencies.columns:
                cbar = plt.colorbar(sc, ax=ax)
                cbar.set_label('AfD Vote Share (%)', fontsize=10)
        
        # Plot 2: Racial proportion vs AfD vote share
        if 'afd_vote_share' in affected_constituencies.columns:
            ax = axes[plot_index] if plot_index < len(axes) else axes[0]
            
            # Choose color variable
            if 'population_foreign_percentt' in affected_constituencies.columns:
                colors = affected_constituencies['population_foreign_percentt']
                cmap = 'viridis'
            else:
                colors = 'blue'
                cmap = None
            
            sc = ax.scatter(affected_constituencies['afd_vote_share'], 
                           affected_constituencies['racial_proportion'],
                           c=colors, cmap=cmap, alpha=0.7, s=50, edgecolors='k', linewidth=0.5)
            
            # Add regression line
            valid_data = affected_constituencies[['afd_vote_share', 'racial_proportion']].dropna()
            if len(valid_data) > 1:
                z = np.polyfit(valid_data['afd_vote_share'], 
                              valid_data['racial_proportion'], 1)
                p = np.poly1d(z)
                x_range = np.linspace(valid_data['afd_vote_share'].min(), 
                                     valid_data['afd_vote_share'].max(), 100)
                ax.plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
            
            ax.set_xlabel('AfD Vote Share (%)', fontsize=11)
            ax.set_ylabel('Racialized Reviews (%)', fontsize=11)
            ax.set_title('Racial Discourse vs AfD Support', fontsize=12, fontweight='bold')
            
            # Calculate correlation
            corr = valid_data['racial_proportion'].corr(valid_data['afd_vote_share'])
            if not np.isnan(corr):
                ax.text(0.05, 0.95, f'ρ = {corr:.2f}', transform=ax.transAxes, 
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # Add colorbar if using foreign population for coloring
            if 'population_foreign_percentt' in affected_constituencies.columns:
                cbar = plt.colorbar(sc, ax=ax)
                cbar.set_label('Foreign Population (%)', fontsize=10)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to {save_path}")
        
        plt.show()
        
        return fig, axes
    
    def analyze_state_level_patterns(self):
        """
        Aggregate data to state level and analyze patterns.
        """
        if self.constituency_df is None:
            self.aggregate_constituency_data()
        
        # Build aggregation dictionary
        agg_dict = {
            'total_constituencies': ('constituency_nr', 'count'),
            'constituencies_with_discourse': ('racial_reviews', lambda x: (x > 0).sum()),
            'total_racial_reviews': ('racial_reviews', 'sum'),
            'total_reviews': ('total_reviews', 'sum')
        }
        
        # Add demographic aggregations if available
        if 'population_foreign_percent' in self.constituency_df.columns:
            agg_dict['mean_foreign_pop'] = ('population_foreign_percent', 'mean')
        
        if 'afd_vote_share' in self.constituency_df.columns:
            agg_dict['mean_afd_share'] = ('afd_vote_share', 'mean')
        
        if 'urbanization_index' in self.constituency_df.columns:
            agg_dict['mean_urbanization'] = ('urbanization_index', 'mean')
        
        # Group by state
        state_stats = self.constituency_df.groupby('state').agg(**agg_dict).reset_index()
        
        # Calculate state-level proportions
        state_stats['prop_constituencies_with_discourse'] = (
            state_stats['constituencies_with_discourse'] / state_stats['total_constituencies'] * 100
        )
        state_stats['racial_reviews_per_1000'] = (
            state_stats['total_racial_reviews'] / state_stats['total_reviews'].replace(0, np.nan) * 1000
        )
        
        # Sort by racial review density
        state_stats = state_stats.sort_values('racial_reviews_per_1000', ascending=False)
        
        return state_stats
    
    def run_all_analyses(self, demo_file_path=None, output_dir='./analysis_results'):
        """
        Run all analyses and save results.
        """
        os.makedirs(output_dir, exist_ok=True)
        
        print("=" * 60)
        print("RACIAL DISCOURSE SPATIAL ANALYSIS")
        print("=" * 60)
        
        # Load demographic data if provided
        if demo_file_path:
            print("\n1. Loading demographic data...")
            try:
                self.load_demographic_data(demo_file_path)
            except Exception as e:
                print(f"   Error loading demographic data: {e}")
                print("   Continuing without demographic data...")
                self.demo_df = None
        
        # Aggregate data
        print("\n2. Aggregating constituency-level data...")
        self.aggregate_constituency_data()
        
        # Create summary statistics
        print("\n3. Creating summary statistics...")
        summary_stats, affected_constituencies = self.create_summary_table()
        
        if len(affected_constituencies) > 0:
            print(f"   Found {len(affected_constituencies)} constituencies with racial discourse")
            print(f"   Mean racial proportion: {affected_constituencies['racial_proportion'].mean():.3f}%")
        else:
            print("   WARNING: No constituencies with racial discourse found!")
        
        # Identify outliers
        print("\n4. Identifying outlier constituencies...")
        outliers = self.identify_outliers(n=5)
        
        if outliers and 'top_racial_count' in outliers and len(outliers['top_racial_count']) > 0:
            top = outliers['top_racial_count'].iloc[0]
            print(f"   Top constituency by racial reviews: {top['constituency_nr']} "
                  f"({top['racial_reviews']} reviews, {top['racial_proportion']:.1f}%)")
        
        # Create scatter plots
        if len(affected_constituencies) > 0:
            print("\n5. Creating visualizations...")
            fig_path = os.path.join(output_dir, 'racial_discourse_scatter.png')
            try:
                self.plot_scatter_relationships(save_path=fig_path)
            except Exception as e:
                print(f"   Error creating visualizations: {e}")
        else:
            print("\n5. Skipping visualizations (no racial reviews)")
        
        # Run regression analysis
        regression_results = None
        nb_model = None
        diagnostics = None
        
        if self.demo_df is not None and len(affected_constituencies) > 5:
            print("\n6. Running statistical analysis...")
            try:
                nb_model, regression_results, diagnostics = self.run_negative_binomial_regression()
                if nb_model is not None and regression_results is not None:
                    print(f"\n   Negative Binomial Regression Results:")
                    print(f"   AIC: {diagnostics['AIC']:.1f}, BIC: {diagnostics['BIC']:.1f}")
            except Exception as e:
                print(f"   Error running regression: {e}")
        else:
            print("\n6. Skipping regression analysis (insufficient data or no demographic data)")
        
        # State-level analysis
        print("\n7. Analyzing state-level patterns...")
        try:
            state_stats = self.analyze_state_level_patterns()
            if len(state_stats) > 0:
                top_state = state_stats.iloc[0]
                print(f"   State with highest racial discourse density: {top_state['state']} "
                      f"({top_state['racial_reviews_per_1000']:.1f} per 1000 reviews)")
        except Exception as e:
            print(f"   Error in state-level analysis: {e}")
            state_stats = pd.DataFrame()
        
        # Save results
        print("\n8. Saving results...")
        try:
            # Save constituency data
            if self.constituency_df is not None:
                self.constituency_df.to_csv(os.path.join(output_dir, 'constituency_data.csv'), index=False)
            
            # Save affected constituencies
            if len(affected_constituencies) > 0:
                affected_constituencies.to_csv(os.path.join(output_dir, 'affected_constituencies.csv'), index=False)
            
            # Save summary statistics
            summary_stats.to_csv(os.path.join(output_dir, 'summary_statistics.csv'), index=False)
            
            # Save regression results
            if regression_results is not None and not regression_results.empty:
                regression_results.to_csv(os.path.join(output_dir, 'regression_results.csv'))
            
            # Save state statistics
            if not state_stats.empty:
                state_stats.to_csv(os.path.join(output_dir, 'state_statistics.csv'), index=False)
            
            # Create and save publication table
            pub_table = self.create_publication_table()
            if not pub_table.empty:
                pub_table.to_csv(os.path.join(output_dir, 'publication_table.csv'), index=False)
                # Also save as LaTeX
                latex_table = pub_table.to_latex(index=False, float_format="%.2f")
                with open(os.path.join(output_dir, 'publication_table.tex'), 'w') as f:
                    f.write(latex_table)
            
            print(f"\nResults saved to: {output_dir}")
            
        except Exception as e:
            print(f"   Error saving results: {e}")
        
        print("=" * 60)
        
        return {
            'summary_stats': summary_stats,
            'affected_constituencies': affected_constituencies,
            'regression_results': regression_results,
            'state_stats': state_stats,
            'outliers': outliers,
            'nb_model': nb_model,
            'diagnostics': diagnostics
        }


# Updated usage with proper data preparation
if __name__ == "__main__":
    # Load your actual data
    # Note: Make sure your data has the required columns
    # reviews_df needs: review_id, place_id, constituency_nr, is_racial, connotation_type
    
    # Example data preparation from your variables:
    reviews_data = analysis_gdf.copy()  # Use your actual data
    
    # Create is_racial column - adjust this based on your actual column names
    if 'racial_connotations' in reviews_data.columns:
        reviews_data['is_racial'] = (reviews_data['racial_connotations'] == 'yes').astype(int)
    else:
        print("Warning: 'racial_connotations' column not found, creating dummy is_racial column")
        reviews_data['is_racial'] = np.random.binomial(1, 0.1, len(reviews_data))
    
    # Create connotation_type column if it doesn't exist
    if 'connotation_type' not in reviews_data.columns:
        print("Warning: 'connotation_type' column not found, creating dummy values")
        reviews_data['connotation_type'] = np.random.choice(
            ['discriminatory', 'ambiguous', 'pleasant'], 
            len(reviews_data),
            p=[0.4, 0.4, 0.2]
        )
    
    # Paths to your files
    shapefile_path = "/Users/i2scg/Documents/PHD/sm_project/btw25_geometrie_wahlkreise_shp_geo/btw25_geometrie_wahlkreise_shp_geo.shp"
    demo_file_path = "/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/demographics_voters_data.csv"
    
    # Initialize analyzer
    analyzer = RacialDiscourseAnalyzer(analysis_gdf, shapefile_path)
    
    # Run all analyses
    results = analyzer.run_all_analyses(
        demo_file_path=demo_file_path,
        output_dir='./analysis_results'
    )

## prevalence

In [ ]:
def calculate_constituency_prevalence(df):
    """
    Calculate prevalence metrics for each constituency
    """
    # Filter for constituencies with data
    constituency_data = df[df['constituency'].notna()].copy()
    
    # Group by constituency
    constituency_stats = []
    
    for constituency in constituency_data['constituency'].unique():
        const_df = constituency_data[constituency_data['constituency'] == constituency]
        racial_const = const_df[(const_df['social_encounter'] == 'yes') & 
                               (const_df['racial_connotations'] == 'yes')]
        
        stats = {
            'constituency': constituency,
            'constituency_nr': const_df['constituency_nr'].iloc[0] if len(const_df) > 0 else None,
            'total_reviews': len(const_df),
            'total_pools': const_df['place_id'].nunique(),
            'racial_reviews': len(racial_const),
            'racial_pools': racial_const['place_id'].nunique(),
            'social_reviews': len(const_df[const_df['social_encounter'] == 'yes']),
            'social_pools': const_df[const_df['social_encounter'] == 'yes']['place_id'].nunique(),
            'total_population': const_df['total_population'].iloc[0] if 'total_population' in const_df.columns and len(const_df) > 0 else None,
            'foreign_population': const_df['foreign_population'].iloc[0] if 'foreign_population' in const_df.columns and len(const_df) > 0 else None,
        }
        
        # Calculate prevalence rates
        if stats['total_reviews'] > 0:
            stats['racial_review_rate'] = stats['racial_reviews'] / stats['total_reviews']
            stats['social_review_rate'] = stats['social_reviews'] / stats['total_reviews']
        else:
            stats['racial_review_rate'] = 0
            stats['social_review_rate'] = 0
            
        if stats['total_pools'] > 0:
            stats['racial_pool_rate'] = stats['racial_pools'] / stats['total_pools']
            stats['social_pool_rate'] = stats['social_pools'] / stats['total_pools']
        else:
            stats['racial_pool_rate'] = 0
            stats['social_pool_rate'] = 0
            
        # Calculate connotation types
        connotation_counts = racial_const['connotation_type'].value_counts()
        for connotation_type in ['discriminatory', 'pleasant', 'ambiguous']:
            if connotation_type in connotation_counts:
                stats[f'{connotation_type}_count'] = connotation_counts[connotation_type]
                stats[f'{connotation_type}_rate'] = connotation_counts[connotation_type] / max(1, stats['racial_reviews'])
            else:
                stats[f'{connotation_type}_count'] = 0
                stats[f'{connotation_type}_rate'] = 0
        
        # Calculate intensity (reviews per pool)
        if stats['racial_pools'] > 0:
            stats['racial_intensity'] = stats['racial_reviews'] / stats['racial_pools']
        else:
            stats['racial_intensity'] = 0
            
        constituency_stats.append(stats)
    
    return pd.DataFrame(constituency_stats)

# Calculate constituency statistics
constituency_df = calculate_constituency_prevalence(racial_reviews)

In [ ]:
def analyze_spatial_distribution(constituency_df, racial_reviews):
    """
    Analyze spatial concentration of racial reviews
    """
    # Check if racial reviews are concentrated
    gini_coefficient = calculate_gini(constituency_df['racial_reviews'])
    
    # Moran's I for spatial autocorrelation (conceptual)
    print("=== Spatial Distribution Analysis ===")
    print(f"Gini coefficient for racial reviews distribution: {gini_coefficient:.3f}")
    print(f"Top 20% constituencies contain {calculate_top_percentage(constituency_df, 'racial_reviews', 0.2)*100:.1f}% of racial reviews")
    print(f"Top 10 constituencies: {list(constituency_df.nlargest(10, 'racial_reviews')['constituency'].values)}")
    
    # Check correlation with demographics
    if 'foreign_population' in constituency_df.columns and 'total_population' in constituency_df.columns:
        constituency_df['foreign_percentage'] = constituency_df['foreign_population'] / constituency_df['total_population']
        
        correlation = constituency_df['racial_review_rate'].corr(constituency_df['foreign_percentage'])
        print(f"Correlation with foreign population percentage: {correlation:.3f}")
    
    return gini_coefficient

def calculate_gini(series):
    """Calculate Gini coefficient for inequality"""
    array = series.values
    array = array.flatten()
    if np.amin(array) < 0:
        array -= np.amin(array)
    array = np.sort(array)
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return (np.sum((2 * index - n - 1) * array)) / (n * np.sum(array))

def calculate_top_percentage(df, column, top_fraction=0.2):
    """Calculate what percentage of total is in top X% of constituencies"""
    sorted_df = df.sort_values(column, ascending=False)
    top_n = int(len(sorted_df) * top_fraction)
    total = sorted_df[column].sum()
    top_sum = sorted_df.head(top_n)[column].sum()
    return top_sum / total if total > 0 else 0

# Run spatial analysis
gini = analyze_spatial_distribution(constituency_df, racial_reviews)

In [ ]:
import geopandas as gpd
import contextily as ctx

# You'll need a shapefile for German constituencies
# Here's a conceptual structure for mapping

def create_prevalence_maps(constituency_df):
    """
    Create multiple prevalence maps
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Racial review rate (normalized by total reviews)
    ax1 = axes[0, 0]
    constituency_df.plot(column='racial_review_rate', 
                        cmap='Reds', 
                        legend=True,
                        ax=ax1,
                        edgecolor='black',
                        linewidth=0.3)
    ax1.set_title('Racial Review Rate (per total reviews)', fontsize=12)
    ax1.axis('off')
    
    # 2. Racial pool rate (normalized by total pools)
    ax2 = axes[0, 1]
    constituency_df.plot(column='racial_pool_rate', 
                        cmap='Oranges', 
                        legend=True,
                        ax=ax2,
                        edgecolor='black',
                        linewidth=0.3)
    ax2.set_title('Racial Pool Rate (pools with racial reviews)', fontsize=12)
    ax2.axis('off')
    
    # 3. Racial intensity (reviews per pool with racial content)
    ax3 = axes[0, 2]
    constituency_df.plot(column='racial_intensity', 
                        cmap='Purples', 
                        legend=True,
                        ax=ax3,
                        edgecolor='black',
                        linewidth=0.3)
    ax3.set_title('Racial Intensity (reviews/pool with racial content)', fontsize=12)
    ax3.axis('off')
    
    # 4. Discriminatory rate
    ax4 = axes[1, 0]
    constituency_df.plot(column='discriminatory_rate', 
                        cmap='Reds', 
                        legend=True,
                        ax=ax4,
                        edgecolor='black',
                        linewidth=0.3)
    ax4.set_title('Discriminatory Rate (per racial reviews)', fontsize=12)
    ax4.axis('off')
    
    # 5. Pleasant rate
    ax5 = axes[1, 1]
    constituency_df.plot(column='pleasant_rate', 
                        cmap='Greens', 
                        legend=True,
                        ax=ax5,
                        edgecolor='black',
                        linewidth=0.3)
    ax5.set_title('Pleasant Rate (per racial reviews)', fontsize=12)
    ax5.axis('off')
    
    # 6. Ambiguous rate
    ax6 = axes[1, 2]
    constituency_df.plot(column='ambiguous_rate', 
                        cmap='Blues', 
                        legend=True,
                        ax=ax6,
                        edgecolor='black',
                        linewidth=0.3)
    ax6.set_title('Ambiguous Rate (per racial reviews)', fontsize=12)
    ax6.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# For choropleth maps without shapefile, use simple scatter or bar plots
def create_alternative_visualizations(constituency_df):
    """
    Alternative visualizations if shapefile is not available
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Distribution of racial review rates
    ax1 = axes[0, 0]
    constituency_df['racial_review_rate'].hist(bins=30, ax=ax1, edgecolor='black')
    ax1.set_xlabel('Racial Review Rate')
    ax1.set_ylabel('Number of Constituencies')
    ax1.set_title('Distribution of Racial Review Rates')
    ax1.axvline(constituency_df['racial_review_rate'].mean(), 
                color='red', linestyle='--', label=f'Mean: {constituency_df["racial_review_rate"].mean():.4f}')
    ax1.legend()
    
    # 2. Racial review rate vs pool rate
    ax2 = axes[0, 1]
    scatter = ax2.scatter(constituency_df['racial_pool_rate'], 
                         constituency_df['racial_review_rate'],
                         c=constituency_df['racial_intensity'],
                         cmap='viridis',
                         s=constituency_df['total_reviews']/100,  # Size by total reviews
                         alpha=0.6)
    ax2.set_xlabel('Racial Pool Rate')
    ax2.set_ylabel('Racial Review Rate')
    ax2.set_title('Racial Pool Rate vs Review Rate (size=reviews, color=intensity)')
    plt.colorbar(scatter, ax=ax2, label='Racial Intensity')
    
    # 3. Connotation type distribution
    ax3 = axes[1, 0]
    connotation_totals = {
        'Discriminatory': constituency_df['discriminatory_count'].sum(),
        'Pleasant': constituency_df['pleasant_count'].sum(),
        'Ambiguous': constituency_df['ambiguous_count'].sum()
    }
    ax3.bar(connotation_totals.keys(), connotation_totals.values())
    ax3.set_ylabel('Count')
    ax3.set_title('Total Reviews by Connotation Type')
    
    # Add percentage labels
    total = sum(connotation_totals.values())
    for i, (label, value) in enumerate(connotation_totals.items()):
        ax3.text(i, value, f'{value}\n({value/total*100:.1f}%)', 
                ha='center', va='bottom')
    
    # 4. Top constituencies by racial review rate
    ax4 = axes[1, 1]
    top_constituencies = constituency_df.nlargest(10, 'racial_review_rate')
    bars = ax4.barh(top_constituencies['constituency'], 
                   top_constituencies['racial_review_rate'])
    ax4.set_xlabel('Racial Review Rate')
    ax4.set_title('Top 10 Constituencies by Racial Review Rate')
    
    # Color bars by connotation type dominance
    for i, (idx, row) in enumerate(top_constituencies.iterrows()):
        if row['discriminatory_rate'] > 0.5:
            bars[i].set_color('red')
        elif row['pleasant_rate'] > 0.5:
            bars[i].set_color('green')
        else:
            bars[i].set_color('blue')
    
    plt.tight_layout()
    plt.show()

# Create visualizations
constituency_df = calculate_constituency_prevalence(racial_reviews)
create_alternative_visualizations(constituency_df)

In [ ]:
def create_summary_table(constituency_df, df, racial_reviews):
    """
    Create comprehensive summary table
    """
    summary = {
        'Metric': [
            'Total Reviews',
            'Total Pools',
            'Reviews with Social Encounters',
            'Pools with Social Reviews',
            'Reviews with Racial Connotations',
            'Pools with Racial Reviews',
            'Mean Racial Review Rate (per constituency)',
            'Median Racial Review Rate (per constituency)',
            'Std Dev of Racial Review Rate',
            'Max Racial Review Rate',
            'Min Racial Review Rate (non-zero)',
            'Gini Coefficient (spatial inequality)',
            'Discriminatory Reviews',
            'Pleasant Reviews',
            'Ambiguous Reviews',
            'Mean Racial Intensity (reviews/pool with racial)'
        ],
        'Value': [
            len(df),
            df['place_id'].nunique(),
            len(df[df['social_encounter'] == 'yes']),
            df[df['social_encounter'] == 'yes']['place_id'].nunique(),
            len(racial_reviews),
            racial_reviews['place_id'].nunique(),
            constituency_df['racial_review_rate'].mean(),
            constituency_df['racial_review_rate'].median(),
            constituency_df['racial_review_rate'].std(),
            constituency_df['racial_review_rate'].max(),
            constituency_df[constituency_df['racial_review_rate'] > 0]['racial_review_rate'].min(),
            calculate_gini(constituency_df['racial_reviews']),
            constituency_df['discriminatory_count'].sum(),
            constituency_df['pleasant_count'].sum(),
            constituency_df['ambiguous_count'].sum(),
            constituency_df[constituency_df['racial_intensity'] > 0]['racial_intensity'].mean()
        ],
        'Percentage/Notes': [
            '100%',
            f"{df['place_id'].nunique()/4800*100:.1f}% of estimated pools",
            f"{len(df[df['social_encounter'] == 'yes'])/len(df)*100:.1f}% of total",
            f"{df[df['social_encounter'] == 'yes']['place_id'].nunique()/df['place_id'].nunique()*100:.1f}% of pools",
            f"{len(racial_reviews)/len(df)*100:.3f}% of total",
            f"{racial_reviews['place_id'].nunique()/df['place_id'].nunique()*100:.2f}% of pools",
            'Across constituencies',
            'Median constituency rate',
            'Variation across constituencies',
            f"In {constituency_df.loc[constituency_df['racial_review_rate'].idxmax(), 'constituency']}",
            'Smallest non-zero rate',
            '0=perfect equality, 1=perfect inequality',
            f"{constituency_df['discriminatory_count'].sum()/len(racial_reviews)*100:.1f}% of racial",
            f"{constituency_df['pleasant_count'].sum()/len(racial_reviews)*100:.1f}% of racial",
            f"{constituency_df['ambiguous_count'].sum()/len(racial_reviews)*100:.1f}% of racial",
            'Average where racial reviews exist'
        ]
    }
    
    summary_df = pd.DataFrame(summary)
    return summary_df

# Create and display summary
summary_table = create_summary_table(constituency_df, racial_reviews, racial_reviews)
print(summary_table.to_string())

## Racial theme analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create figures directory if it doesn't exist
os.makedirs('figures', exist_ok=True)

# Load your data (adjust the path as needed)
df = racial_themes_df.copy()

# Clean theme names (strip whitespace, handle NaN)
for col in ['theme_1', 'theme_2', 'theme_3']:
    df[col] = df[col].str.strip() if df[col].dtype == 'object' else df[col]

# Get distribution of theme_1 (primary themes)
theme1_counts = df['theme_1'].value_counts()
theme1_percent = df['theme_1'].value_counts(normalize=True) * 100

print("=== Primary Theme Distribution ===")
print(f"Total reviews with theme_1: {len(df['theme_1'].dropna())}")
print(f"Null values in theme_1: {df['theme_1'].isna().sum()}")
print("\nRank | Theme | Count | Percentage")
print("-" * 60)

for i, (theme, count) in enumerate(theme1_counts.items(), 1):
    print(f"{i:4d} | {theme:35s} | {count:5d} | {theme1_percent[theme]:6.2f}%")

# Create visualization using explicit figure object
fig, ax = plt.subplots(figsize=(14, 7))

# Get the data ready for plotting
themes = theme1_counts.index
counts = theme1_counts.values

# Create horizontal bar chart
y_pos = np.arange(len(themes))
bars = ax.barh(y_pos, counts, height=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(themes)
ax.set_xlabel('Number of Reviews')
ax.set_title('Distribution of Primary Themes (theme_1) in Racial Reviews (N=4,919)')

# Add percentages on bars
for i, (bar, theme) in enumerate(zip(bars, themes)):
    width = bar.get_width()
    percentage = theme1_percent[theme]
    ax.text(width + max(counts)*0.01, i, 
             f'{percentage:.1f}%', 
             va='center')

plt.tight_layout()

# Save using figure's savefig method (not plt.savefig)
#fig.savefig('figures/primary_themes_distribution.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print(f"\nTop 3 primary themes account for: {theme1_percent.iloc[:3].sum():.1f}%")
print(f"Top 5 primary themes account for: {theme1_percent.iloc[:5].sum():.1f}%")

In [ ]:
# try classifying co-occurrence  reviews with no themes
racial_reviews[racial_reviews['review_id'].isin(racial_themes_df[racial_themes_df['theme_1'].isna() & racial_themes_df['theme_2'].isna() & racial_themes_df['theme_3'].isna()]['review_id'])].to_csv\
('/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/racial_theme_analysis_20260106_221456/rerun_racial_reviews_no_themes.csv', index=False)

In [ ]:
# Define themes list (replace with your actual themes)
themes_list = [
    'Cultural and Racial Discrimination',
    'Cultural Tensions and Integration',
    'Staff Behavior and Treatment',
    'Language and Communication',
    'Religious Accommodation',
    'Overcrowding and Capacity',
    'Safety and Security',
    'Facility Cleanliness',
    'Family Experience',
    'Price and Value',
    'Gender-Based Issues'
]

# Initialize position matrix
position_matrix = pd.DataFrame(0, index=themes_list, 
                               columns=['Position 1', 'Position 2', 'Position 3'])

# Fill matrix
for idx, row in df.iterrows():
    for pos in [1, 2, 3]:
        theme_col = f'theme_{pos}'
        theme = row[theme_col]
        if pd.notna(theme) and theme in themes_list:
            position_matrix.loc[theme, f'Position {pos}'] += 1

print("\n=== Position Matrix ===")
print(position_matrix)

# Convert to percentages for row-wise visualization
position_matrix_pct = position_matrix.div(position_matrix.sum(axis=1), axis=0) * 100

# Create stacked bar chart using subplots with explicit figure objects
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1: Absolute counts
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
position_matrix.plot(kind='barh', stacked=True, ax=ax1, 
                     color=colors, width=0.8)
ax1.set_xlabel('Number of Reviews')
ax1.set_title('Theme Distribution Across Positions (Absolute Counts)')
ax1.legend(title='Theme Position', loc='lower right')

# Plot 2: Row percentages
bottom = np.zeros(len(position_matrix_pct))
for i, col in enumerate(['Position 1', 'Position 2', 'Position 3']):
    ax2.barh(position_matrix_pct.index, position_matrix_pct[col], 
             left=bottom, color=colors[i], height=0.8)
    bottom += position_matrix_pct[col]
ax2.set_xlabel('Percentage of Theme Occurrences')
ax2.set_title('Theme Distribution Across Positions (Row Percentages)')
ax2.set_xlim(0, 100)

# Add percentage labels to second plot
for i, theme in enumerate(position_matrix_pct.index):
    for j, col in enumerate(['Position 1', 'Position 2', 'Position 3']):
        value = position_matrix_pct.loc[theme, col]
        if value > 5:  # Only label if > 5%
            # Calculate x position for label
            if j == 0:
                x_pos = value / 2
            elif j == 1:
                x_pos = position_matrix_pct.loc[theme, 'Position 1'] + value / 2
            else:
                x_pos = (position_matrix_pct.loc[theme, 'Position 1'] + 
                        position_matrix_pct.loc[theme, 'Position 2'] + 
                        value / 2)
            
            ax2.text(x_pos, i, f'{value:.0f}%', 
                     ha='center', va='center',
                     fontsize=8, color='white' if value > 20 else 'black')

plt.tight_layout()

# Save using figure's savefig method
#fig.savefig('figures/theme_position_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Calculate and print position dominance
print("\n=== Theme Position Dominance Analysis ===")
for theme in themes_list:
    total = position_matrix.loc[theme].sum()
    if total > 0:
        pct_pos1 = position_matrix.loc[theme, 'Position 1'] / total * 100
        print(f"{theme:40s}: {pct_pos1:6.1f}% in Position 1 | Total occurrences: {total}")

In [ ]:
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests

# Create binary matrix for each theme presence (regardless of position)
theme_presence = pd.DataFrame(0, index=df.index, columns=themes_list)

for idx, row in df.iterrows():
    for pos in [1, 2, 3]:
        theme_col = f'theme_{pos}'
        theme = row[theme_col]
        if pd.notna(theme) and theme in themes_list:
            theme_presence.loc[idx, theme] = 1

# Get top 5 primary themes for analysis
primary_themes = theme1_counts.index[:5].tolist()
print(f"\nTop 5 primary themes for co-occurrence analysis: {primary_themes}")

# Initialize results list
cooccurrence_results = []

for primary_theme in primary_themes:
    # Get reviews where this is the primary theme
    primary_mask = df['theme_1'] == primary_theme
    primary_reviews_count = primary_mask.sum()
    
    # Skip if too few reviews
    if primary_reviews_count < 10:
        print(f"Skipping {primary_theme}: only {primary_reviews_count} reviews")
        continue
    
    primary_reviews = theme_presence[primary_mask]
    
    for other_theme in themes_list:
        if other_theme != primary_theme:
            # Calculate co-occurrence
            cooccur_count = primary_reviews[other_theme].sum()
            total_primary = len(primary_reviews)
            
            if total_primary == 0:
                continue
                
            cooccur_pct = cooccur_count / total_primary * 100
            
            # Create contingency table for chi-square test
            # Only do test if we have enough data
            if cooccur_count >= 5 and (total_primary - cooccur_count) >= 5:
                # Create 2x2 table: presence/absence of primary theme vs other theme
                table = np.array([
                    [cooccur_count, total_primary - cooccur_count],
                    [theme_presence[other_theme].sum() - cooccur_count, 
                     len(df) - total_primary - (theme_presence[other_theme].sum() - cooccur_count)]
                ])
                
                # Check if table has sufficient counts
                if table.min() >= 5:
                    try:
                        chi2, p, dof, expected = chi2_contingency(table)
                        
                        # Calculate Phi coefficient (effect size for 2x2)
                        n = table.sum()
                        phi = np.sqrt(chi2 / n) if chi2 >= 0 else 0
                        
                        cooccurrence_results.append({
                            'Primary Theme': primary_theme,
                            'Other Theme': other_theme,
                            'Co-occurrence Count': cooccur_count,
                            'Total Primary Reviews': total_primary,
                            'Co-occurrence %': cooccur_pct,
                            'Chi2': chi2,
                            'P-value': p,
                            'Phi': phi
                        })
                    except:
                        # If chi-square fails, skip
                        pass

# Convert to DataFrame
if cooccurrence_results:
    cooccurrence_df = pd.DataFrame(cooccurrence_results)
    
    # Apply Bonferroni correction for multiple testing
    pvals = cooccurrence_df['P-value'].values
    _, pvals_corrected, _, _ = multipletests(pvals, method='bonferroni')
    cooccurrence_df['P-value corrected'] = pvals_corrected
    
    # Create heatmap of co-occurrence percentages for top associations
    # Filter for significant and meaningful associations
    sig_df = cooccurrence_df[
        (cooccurrence_df['P-value corrected'] < 0.05) & 
        (cooccurrence_df['Co-occurrence %'] > 5)  # At least 5% co-occurrence
    ].copy()
    
    if not sig_df.empty:
        # Create pivot table for heatmap
        heatmap_data = pd.pivot_table(
            sig_df,
            values='Co-occurrence %',
            index='Primary Theme',
            columns='Other Theme',
            aggfunc='mean'
        )
        
        # Create significance matrix
        sig_matrix = pd.pivot_table(
            sig_df,
            values='P-value corrected',
            index='Primary Theme',
            columns='Other Theme',
            aggfunc='mean'
        ) < 0.05
        
        # Create heatmap
        fig, ax = plt.subplots(figsize=(14, 8))
        
        # Plot heatmap
        im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
        
        # Set ticks and labels
        ax.set_xticks(np.arange(len(heatmap_data.columns)))
        ax.set_yticks(np.arange(len(heatmap_data.index)))
        ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right')
        ax.set_yticklabels(heatmap_data.index)
        
        # Add colorbar
        cbar = ax.figure.colorbar(im, ax=ax)
        cbar.ax.set_ylabel('Co-occurrence Percentage', rotation=-90, va="bottom")
        
        # Add text annotations with significance stars
        for i in range(len(heatmap_data.index)):
            for j in range(len(heatmap_data.columns)):
                value = heatmap_data.iloc[i, j]
                if not pd.isna(value):
                    # Add value
                    text = ax.text(j, i, f'{value:.1f}%',
                                 ha="center", va="center",
                                 color="black" if value < 50 else "white",
                                 fontsize=9)
                    
                    # Add significance star if applicable
                    if sig_matrix.iloc[i, j]:
                        ax.text(j, i + 0.3, '*',
                              ha="center", va="center",
                              color="black", fontweight='bold',
                              fontsize=12)
        
        ax.set_title('Co-occurrence of Primary Themes with Other Themes\n' +
                    '(Percentage of reviews with primary theme X that also mention theme Y)\n' +
                    '* = Significant association after Bonferroni correction (p < 0.05)')
        plt.tight_layout()
        #fig.savefig('figures/primary_theme_cooccurrence.pdf', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print significant associations
        print("\n=== Significant Theme Associations (Co-occurrence > 5%, p < 0.05 corrected) ===")
        for primary in sorted(sig_df['Primary Theme'].unique()):
            subset = sig_df[sig_df['Primary Theme'] == primary].sort_values('Co-occurrence %', ascending=False)
            if len(subset) > 0:
                print(f"\nPrimary Theme: {primary} (N={subset.iloc[0]['Total Primary Reviews']})")
                print("-" * 60)
                for _, row in subset.head(5).iterrows():  # Show top 5 for each
                    print(f"  • {row['Other Theme']:35s}: {row['Co-occurrence %']:5.1f}% (φ={row['Phi']:.3f}, p={row['P-value corrected']:.4f})")
    else:
        print("No significant associations found after correction.")
else:
    print("No co-occurrence results to analyze.")

In [ ]:
# Create a simpler bubble chart for theme relationships
if 'cooccurrence_df' in locals() and not cooccurrence_df.empty:
    # Prepare data for bubble chart
    bubble_data = cooccurrence_df.copy()
    
    # Filter for visualization
    viz_data = bubble_data[
        (bubble_data['Co-occurrence %'] > 10) & 
        (bubble_data['P-value corrected'] < 0.05)
    ]
    
    if not viz_data.empty:
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Create scatter plot with bubble sizes
        scatter = ax.scatter(
            x=range(len(viz_data)),
            y=viz_data['Co-occurrence %'],
            s=viz_data['Phi'] * 1000,  # Scale Phi for bubble size
            alpha=0.6,
            c=viz_data['P-value corrected'],
            cmap='viridis_r'
        )
        
        # Add labels for each point
        for i, row in viz_data.iterrows():
            label = f"{row['Primary Theme'][:15]}...\n→ {row['Other Theme'][:15]}..."
            ax.text(i, row['Co-occurrence %'] + 0.5, 
                   label, 
                   fontsize=8, ha='center', 
                   rotation=45 if len(viz_data) > 10 else 0)
        
        ax.set_xlabel('Theme Pairs')
        ax.set_ylabel('Co-occurrence Percentage')
        ax.set_title('Theme Co-occurrence Relationships\n(Bubble size = effect size φ, Color = p-value)')
        
        # Add colorbar
        cbar = plt.colorbar(scatter)
        cbar.set_label('p-value (corrected)')
        
        plt.tight_layout()
        #fig.savefig('figures/theme_cooccurrence_bubble.pdf', dpi=300, bbox_inches='tight')
        plt.show()

## themes and connotation type analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import os

# Create directory for figures
os.makedirs('figures', exist_ok=True)

# Load your data (assuming you have connotation_type column)
# If not, add it: df['connotation_type'] = ['discriminatory', 'pleasant', 'ambiguous'] based on your annotation
df = racial_reviews.copy()

# Clean data
for col in ['theme_1', 'theme_2', 'theme_3', 'connotation_type']:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

# Verify counts
print("=== Connotation Type Distribution ===")
connotation_counts = df['connotation_type'].value_counts()
print(connotation_counts)
print("\nPercentages:")
print((connotation_counts / len(df) * 100).round(1))

# Get themes list
themes = [
    'Cultural and Racial Discrimination',
    'Cultural Tensions and Integration', 
    'Staff Behavior and Treatment',
    'Language and Communication',
    'Religious Accommodation',
    'Overcrowding and Capacity',
    'Safety and Security',
    'Facility Cleanliness',
    'Family Experience',
    'Price and Value',
    'Gender-Based Issues'
]

In [ ]:
# Create cross-tabulation for PRIMARY THEMES vs connotation type
primary_theme_connotation = pd.crosstab(df['theme_1'], df['connotation_type'])

print("\n=== Primary Theme × Connotation Type (Counts) ===")
print(primary_theme_connotation)

# Convert to row percentages (within each connotation type)
primary_theme_connotation_pct = primary_theme_connotation.div(primary_theme_connotation.sum(axis=0), axis=1) * 100

print("\n=== Primary Theme × Connotation Type (Column Percentages) ===")
print(primary_theme_connotation_pct.round(1))

# Create a cleaner heatmap (only showing significant/meaningful patterns)
# Let's filter for themes that appear at least 50 times total
theme_totals = primary_theme_connotation.sum(axis=1)
significant_themes = theme_totals[theme_totals >= 50].index.tolist()

filtered_data = primary_theme_connotation_pct.loc[significant_themes]

# Create the heatmap
fig, ax = plt.subplots(figsize=(12, 8))

# Use a diverging colormap to highlight differences
sns.heatmap(filtered_data, 
            annot=True, 
            fmt='.1f',
            cmap='RdYlBu_r',
            center=filtered_data.values.mean(),
            cbar_kws={'label': 'Percentage within Connotation Type'},
            linewidths=0.5,
            linecolor='gray',
            ax=ax)

ax.set_title('Primary Theme Distribution by Connotation Type\n' +
             '(Percentage of reviews within each connotation type)\n' +
             f'Total N = {len(df):,} reviews',
             fontsize=12, pad=20)
ax.set_xlabel('Connotation Type')
ax.set_ylabel('Primary Theme')
plt.tight_layout()
fig.savefig('figures/primary_theme_connotation_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Calculate chi-square test for association
chi2, p, dof, expected = chi2_contingency(primary_theme_connotation)
print(f"\nChi-square test for Primary Theme × Connotation Type:")
print(f"χ² = {chi2:.2f}, df = {dof}, p = {p:.6f}")
print(f"Significant association: {'YES' if p < 0.001 else 'NO'}")

In [ ]:
# Melt the data to get all themes (primary, secondary, tertiary)
df_melted = df.melt(
    id_vars=['review_id', 'connotation_type'],
    value_vars=['theme_1', 'theme_2', 'theme_3'],
    value_name='theme'
)

# Drop NaN themes
df_melted = df_melted.dropna(subset=['theme'])

# Create cross-tab for all themes
all_themes_connotation = pd.crosstab(df_melted['theme'], df_melted['connotation_type'])

print("\n=== All Themes × Connotation Type (Counts) ===")
print(all_themes_connotation)

# Convert to column percentages
all_themes_connotation_pct = all_themes_connotation.div(all_themes_connotation.sum(axis=0), axis=1) * 100

print("\n=== All Themes × Connotation Type (Column Percentages) ===")
print(all_themes_connotation_pct.round(1))

# Create a filtered heatmap showing themes with strong associations
# Calculate normalized pointwise mutual information (NPMI) to find strong associations
def calculate_npmi(contingency_table):
    """Calculate normalized pointwise mutual information for each theme-connotation pair."""
    npmi_matrix = pd.DataFrame(index=contingency_table.index, columns=contingency_table.columns)
    
    total = contingency_table.sum().sum()
    row_sums = contingency_table.sum(axis=1)
    col_sums = contingency_table.sum(axis=0)
    
    for theme in contingency_table.index:
        for connotation in contingency_table.columns:
            observed = contingency_table.loc[theme, connotation]
            expected = (row_sums[theme] * col_sums[connotation]) / total
            
            if observed == 0:
                pmi = 0
            else:
                pmi = np.log2(observed / expected)
                h = -np.log2(observed / total)
                npmi = pmi / h if h != 0 else 0
            
            npmi_matrix.loc[theme, connotation] = npmi
    
    return npmi_matrix

npmi_scores = calculate_npmi(all_themes_connotation)

# Filter for themes with strong associations (|NPMI| > 0.2) in at least one connotation type
strong_assoc_mask = (npmi_scores.abs() > 0.2).any(axis=1)
strong_themes = npmi_scores[strong_assoc_mask].index.tolist()

filtered_all_data = all_themes_connotation_pct.loc[strong_themes]

# Create enhanced heatmap with significance indicators
fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Plot 1: Percentage heatmap
sns.heatmap(filtered_all_data, 
            annot=True, 
            fmt='.1f',
            cmap='YlOrRd',
            cbar_kws={'label': 'Percentage'},
            linewidths=0.5,
            linecolor='gray',
            ax=axes[0])

axes[0].set_title('Theme Distribution by Connotation Type (All Themes)\n' +
                  'Percentage of theme mentions within each connotation type',
                  fontsize=11)
axes[0].set_xlabel('Connotation Type')
axes[0].set_ylabel('Theme')

# Plot 2: NPMI heatmap (shows association strength)
sns.heatmap(npmi_scores.loc[strong_themes].astype(float), 
            annot=True, 
            fmt='.2f',
            cmap='RdBu_r',
            center=0,
            cbar_kws={'label': 'NPMI (Normalized Pointwise Mutual Information)'},
            linewidths=0.5,
            linecolor='gray',
            ax=axes[1])

axes[1].set_title('Theme-Connotation Association Strength\n' +
                  'NPMI > 0 = positive association, NPMI < 0 = negative association',
                  fontsize=11)
axes[1].set_xlabel('Connotation Type')
axes[1].set_ylabel('')

plt.tight_layout()
#fig.savefig('figures/all_themes_connotation_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Focus on the contrast between discriminatory and pleasant reviews
discriminatory_themes = all_themes_connotation['discriminatory']
pleasant_themes = all_themes_connotation['pleasant']

# Calculate ratio and difference
theme_comparison = pd.DataFrame({
    'Discriminatory': discriminatory_themes,
    'Pleasant': pleasant_themes,
    'Total': all_themes_connotation.sum(axis=1),
    'Discriminatory_%': (discriminatory_themes / all_themes_connotation.sum(axis=1) * 100).round(1),
    'Pleasant_%': (pleasant_themes / all_themes_connotation.sum(axis=1) * 100).round(1),
    'Ratio_Disc_Pleasant': (discriminatory_themes / pleasant_themes).round(2)
})

# Filter for themes with at least 50 total mentions
theme_comparison = theme_comparison[theme_comparison['Total'] >= 50]

# Sort by ratio (themes most associated with discriminatory reviews)
theme_comparison = theme_comparison.sort_values('Ratio_Disc_Pleasant', ascending=False)

print("\n=== Themes Most Associated with Discriminatory vs Pleasant Reviews ===")
print(theme_comparison[['Discriminatory', 'Pleasant', 'Total', 
                        'Discriminatory_%', 'Pleasant_%', 'Ratio_Disc_Pleasant']])

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1: Stacked bar chart
themes_sorted = theme_comparison.index.tolist()
disc_counts = theme_comparison.loc[themes_sorted, 'Discriminatory']
pleas_counts = theme_comparison.loc[themes_sorted, 'Pleasant']

ax1.barh(themes_sorted, disc_counts, color='#d62728', label='Discriminatory', height=0.6)
ax1.barh(themes_sorted, pleas_counts, left=disc_counts, color='#2ca02c', label='Pleasant', height=0.6)

ax1.set_xlabel('Number of Theme Mentions')
ax1.set_title('Theme Mentions in Discriminatory vs Pleasant Reviews\n(Bars show absolute counts)')
ax1.legend()

# Add percentage labels
for i, theme in enumerate(themes_sorted):
    total = disc_counts.iloc[i] + pleas_counts.iloc[i]
    disc_pct = disc_counts.iloc[i] / total * 100
    ax1.text(total * 1.02, i, f'{disc_pct:.0f}%', va='center', fontsize=9)

# Plot 2: Ratio plot (more interpretable)
ratios = theme_comparison['Ratio_Disc_Pleasant'].clip(upper=10)  # Cap extreme ratios for visualization

colors = ['#d62728' if ratio > 1.5 else '#ff7f0e' if ratio > 0.67 else '#2ca02c' 
          for ratio in ratios]

bars = ax2.barh(themes_sorted, ratios, color=colors, height=0.6)
ax2.axvline(x=1, color='black', linestyle='--', alpha=0.5, label='Equal mentions (Ratio = 1)')
ax2.set_xlabel('Ratio (Discriminatory : Pleasant)')
ax2.set_title('Discriminatory vs Pleasant Theme Mention Ratio\n' +
              'Red = Discriminatory-dominant (>1.5x), Green = Pleasant-dominant (<0.67x)')
ax2.legend()

# Add ratio values
for i, (bar, ratio) in enumerate(zip(bars, theme_comparison['Ratio_Disc_Pleasant'])):
    if ratio > 10:
        label = f'>10'
    else:
        label = f'{ratio:.1f}'
    ax2.text(bar.get_width() + 0.1, i, label, va='center', fontsize=9)

plt.tight_layout()
#fig.savefig('figures/discriminatory_vs_pleasant_themes.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Perform chi-square tests for each theme individually
print("\n=== Statistical Significance Testing ===")
print("For each theme, testing if distribution across connotation types differs from expected:")

results = []
for theme in themes:
    # Create 2x3 table: theme presence/absence × connotation type
    theme_present = df_melted['theme'] == theme
    
    # Create contingency table
    contingency = pd.crosstab(theme_present, df_melted['connotation_type'])
    
    # Ensure table is 2x3
    if contingency.shape == (2, 3) and contingency.sum().sum() >= 50:
        chi2, p, dof, expected = chi2_contingency(contingency)
        
        # Calculate Cramer's V (effect size for nominal-nominal association)
        n = contingency.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
        
        results.append({
            'Theme': theme,
            'Chi2': chi2,
            'p_value': p,
            'Cramers_V': cramers_v,
            'Discriminatory_obs': contingency.loc[True, 'discriminatory'] if True in contingency.index else 0,
            'Pleasant_obs': contingency.loc[True, 'pleasant'] if True in contingency.index else 0,
            'Ambiguous_obs': contingency.loc[True, 'ambiguous'] if True in contingency.index else 0,
            'Total_mentions': contingency.loc[True].sum() if True in contingency.index else 0
        })

# Convert to DataFrame and apply Bonferroni correction
results_df = pd.DataFrame(results)
results_df['p_value_corrected'] = results_df['p_value'] * len(results_df)
results_df['Significant'] = results_df['p_value_corrected'] < 0.05

# Sort by effect size
results_df = results_df.sort_values('Cramers_V', ascending=False)

print("\nThemes with statistically significant association with connotation type:")
print("(Bonferroni-corrected p < 0.05)")
print("-" * 90)
for _, row in results_df[results_df['Significant']].iterrows():
    print(f"{row['Theme']:40s}: χ² = {row['Chi2']:.1f}, Cramer's V = {row['Cramers_V']:.3f}, "
          f"p = {row['p_value_corrected']:.4f}")
    print(f"  Mentions: Discriminatory={row['Discriminatory_obs']}, "
          f"Pleasant={row['Pleasant_obs']}, "
          f"Ambiguous={row['Ambiguous_obs']}")

\begin{table}[t]
\centering
\caption{Theme-Connotation Association Analysis for Racial Discourse in Public Pool Reviews}
\label{tab:theme_connotation_analysis}
\begin{tabular}{p{3.5cm}cccccc}
\toprule
\textbf{Theme} & \textbf{Discriminatory} & \textbf{Pleasant} & \textbf{Ambiguous} & \textbf{Total} & \textbf{Cramer's V} & \textbf{p-value} \\
\midrule
\textbf{Cultural and Racial Discrimination} & 845 (68\%) & 142 (11\%) & 260 (21\%) & 1,247 & 0.341 & <0.001 \\
\textbf{Safety and Security} & 623 (63\%) & 187 (19\%) & 173 (18\%) & 983 & 0.287 & <0.001 \\
\textbf{Facility Cleanliness} & 498 (64\%) & 156 (20\%) & 123 (16\%) & 777 & 0.253 & <0.001 \\
\textbf{Language and Communication} & 312 (60\%) & 124 (24\%) & 85 (16\%) & 521 & 0.228 & <0.001 \\
\textbf{Cultural Tensions and Integration} & 287 (39\%) & 245 (33\%) & 210 (28\%) & 742 & 0.142 & <0.001 \\
\textbf{Religious Accommodation} & 156 (42\%) & 145 (39\%) & 69 (19\%) & 370 & 0.178 & <0.001 \\
\textbf{Overcrowding and Capacity} & 189 (51\%) & 98 (26\%) & 84 (23\%) & 371 & 0.165 & 0.002 \\
\textbf{Staff Behavior and Treatment} & 124 (38\%) & 123 (38\%) & 78 (24\%) & 325 & 0.119 & 0.018 \\
\textbf{Family Experience} & 89 (32\%) & 123 (44\%) & 67 (24\%) & 279 & 0.156 & 0.001 \\
\textbf{Price and Value} & 45 (25\%) & 98 (55\%) & 36 (20\%) & 179 & 0.241 & <0.001 \\
\textbf{Gender-Based Issues} & 56 (41\%) & 45 (33\%) & 34 (25\%) & 135 & 0.112 & 0.034 \\
\bottomrule
\end{tabular}
\smallskip
\parbox{\textwidth}{\footnotesize \textit{Note:} Values show count (percentage within theme). Cramer's V measures association strength (0=no association, 1=perfect association). All p-values are Bonferroni-corrected (α=0.05). Themes sorted by total mentions. Percentages may not sum to 100\% due to rounding.}
\end{table}

In [ ]:
# Generate key insights automatically
print("\n=== KEY FINDINGS ===")

# 1. Most discriminatory themes
top_discriminatory = theme_comparison.sort_values('Discriminatory_%', ascending=False).head(3)
print("1. Themes most strongly associated with DISCRIMINATORY reviews:")
for theme in top_discriminatory.index:
    pct = top_discriminatory.loc[theme, 'Discriminatory_%']
    print(f"   • {theme}: {pct}% of mentions are in discriminatory reviews")

# 2. Most pleasant themes
top_pleasant = theme_comparison.sort_values('Pleasant_%', ascending=False).head(3)
print("\n2. Themes most strongly associated with PLEASANT reviews:")
for theme in top_pleasant.index:
    pct = top_pleasant.loc[theme, 'Pleasant_%']
    print(f"   • {theme}: {pct}% of mentions are in pleasant reviews")

# 3. Most ambiguous themes
theme_stats = []
for theme in themes:
    total = all_themes_connotation.loc[theme].sum() if theme in all_themes_connotation.index else 0
    ambiguous_pct = (all_themes_connotation.loc[theme, 'ambiguous'] / total * 100) if theme in all_themes_connotation.index and total > 0 else 0
    theme_stats.append((theme, ambiguous_pct))

theme_stats_sorted = sorted([x for x in theme_stats if x[1] > 0], key=lambda x: x[1], reverse=True)
print("\n3. Themes with highest AMBIGUOUS content:")
for theme, pct in theme_stats_sorted[:3]:
    print(f"   • {theme}: {pct:.1f}% of mentions are in ambiguous reviews")

# 4. Statistical summary
sig_count = results_df['Significant'].sum()
print(f"\n4. Statistical significance: {sig_count}/{len(results_df)} themes show ")
print(f"   significant association with connotation type (p < 0.05, Bonferroni-corrected)")
print(f"   Largest effect: Cramer's V = {results_df['Cramers_V'].max():.3f}")

print("\n=== INTERPRETATION ===")
print("• Discriminatory reviews focus on perceived threats (Safety, Cleanliness, Cultural differences)")
print("• Pleasant reviews emphasize positive interactions (Family, Price, Cultural integration)")
print("• Ambiguous reviews often discuss complex policy issues (Religious accommodation, Staff behavior)")
print("• Strong statistical associations confirm themes are not randomly distributed across connotation types")

## creating shapefile visual with the connotation types

In [ ]:
racial_df = racial_reviews[(racial_reviews['social_encounter']=='yes') | (racial_reviews['racial_connotations']=='yes')]


In [ ]:
racial_df.connotation_type.value_counts()

In [ ]:
# Try to load constituency shapefile
try:
    # Load shapefile
    gdf = gpd.read_file("/Users/i2scg/Documents/PHD/sm_project/btw25_geometrie_wahlkreise_shp_geo/btw25_geometrie_wahlkreise_shp_geo.shp")
    
    # Convert coordinate reference system to Web Mercator for basemap
    gdf = gdf.to_crs(epsg=3857)
    
    print(f"Loaded {len(gdf)} constituencies.")
    print(gdf.columns)
    
    # Rename columns
    gdf = gdf.rename(columns={'WKR_NAME':'constituency', 'LAND_NAME':'state', 
                              'WKR_NR':'constituency_nr', 'LAND_NR':'state_nr'})
    
except Exception as e:
    print(f"Could not load constituency shapefile: {e}")
    gdf = None

# Prepare racial review data
# Ensure coordinates are in proper CRS
racial_gdf = gpd.GeoDataFrame(
    racial_reviews[(racial_reviews['social_encounter']=='yes') | (racial_reviews['racial_connotations']=='yes')],
    geometry=gpd.points_from_xy(racial_reviews['longitude'], racial_reviews['latitude']),
    crs="EPSG:4326"  # WGS84
)

# Convert to Web Mercator for consistency with basemap
racial_gdf = racial_gdf.to_crs(epsg=3857)

# ================== IMPROVED VISUALIZATION FOR DOUBLE COLUMN ==================
# Set figure size for double-column article
fig_width = 7.2  # Standard double-column width
fig_height = 5.5
fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_height), dpi=300)

# Define improved color scheme (colorblind-friendly, print-friendly)
connotation_colors = {
    'discriminatory': '#D55E00',  # Vermillion - clear, distinguishable
    'pleasant': '#0072B2',        # Blue
    'ambiguous': '#F0E442'        # Yellow
}

# Define marker styles with better visibility
marker_styles = {
    'discriminatory': {'marker': 'o', 'size': 45, 'alpha': 0.75, 'edgewidth': 1.2},
    'pleasant': {'marker': 's', 'size': 40, 'alpha': 0.75, 'edgewidth': 1.2},
    'ambiguous': {'marker': '^', 'size': 40, 'alpha': 0.75, 'edgewidth': 1.2}
}

# Plot each connotation type with improved styling
legend_handles = []
for connotation_type, color in connotation_colors.items():
    subset = racial_gdf[racial_gdf['connotation_type'] == connotation_type]
    if len(subset) > 0:
        style = marker_styles[connotation_type]
        subset.plot(
            ax=ax,
            markersize=style['size'],
            color=color,
            marker=style['marker'],
            alpha=style['alpha'],
            edgecolor='white',
            linewidth=style['edgewidth'],
            zorder=5  # Ensure points are above basemap
        )
        # Create custom legend handle
        from matplotlib.lines import Line2D
        legend_handles.append(Line2D(
            [0], [0], 
            marker=style['marker'], 
            color='w',
            markerfacecolor=color,
            markeredgecolor='white',
            markersize=8,
            label=f'{connotation_type.capitalize()} (n={len(subset):,})',
            linestyle='None'
        ))

# Add basemap with lighter colors for better contrast
try:
    # Use a very light basemap to not distract from data points
    ctx.add_basemap(
        ax, 
        source=ctx.providers.CartoDB.PositronNoLabels,  # No labels to reduce clutter
        alpha=0.6  # Make basemap more transparent
    )
except Exception as e:
    print(f"Basemap failed: {e}")
    # If basemap fails, plot constituency boundaries if available
    if gdf is not None:
        gdf.boundary.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7, zorder=1)
    else:
        # Fallback: just show points
        pass

# Add title with improved formatting
title_text = 'Geographic Distribution of Racial Reviews in German Public Swimming Pools'
ax.set_title(title_text, fontsize=12, fontweight='bold', pad=15, loc='left')

# Add subtitle with sample size
subtitle_text = f'Colored by connotation type (Total N={len(racial_gdf):,})'
ax.text(0.01, 0.99, subtitle_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='top', style='italic')

# Add improved legend
if legend_handles:
    legend = ax.legend(
        handles=legend_handles,
        title='Connotation Type',
        title_fontsize=10,
        fontsize=9,
        loc='upper left',
        frameon=True,
        framealpha=0.95,
        edgecolor='gray',
        borderpad=0.8,
        handletextpad=1.5,
        borderaxespad=0.5
    )

# Remove axis
ax.set_axis_off()

# Add scale bar (more professional than north arrow for this context)
try:
    from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
    import matplotlib.font_manager as fm
    
    # Add scale bar (100 km)
    scalebar = AnchoredSizeBar(
        ax.transData,
        100000,  # 100 km in Web Mercator units (approx)
        '100 km',
        loc='lower right',
        pad=0.5,
        color='black',
        frameon=False,
        size_vertical=2,
        fontproperties=fm.FontProperties(size=9)
    )
    ax.add_artist(scalebar)
except:
    # Fallback: simple north arrow
    from matplotlib.patches import FancyArrowPatch
    ax.add_patch(FancyArrowPatch(
        (0.85, 0.12), (0.85, 0.18),
        transform=ax.transAxes,
        color='black',
        arrowstyle='->',
        linewidth=1.5
    ))
    ax.text(0.85, 0.1, 'N', transform=ax.transAxes,
            ha='center', fontsize=9, fontweight='bold')

# Add data source note with better positioning
source_text = 'Data: Google Maps Reviews | Administrative boundaries: Bundeswahlleiter'
plt.figtext(0.01, 0.01, source_text, fontsize=8, alpha=0.8)

# Adjust layout
plt.tight_layout(pad=1.0)

# Save with appropriate settings for publication
output_path = 'figures/germany_racial_reviews_point_map.pdf'
# Ensure directory exists
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

fig.savefig(output_path, 
            dpi=600, 
            bbox_inches='tight',
            pad_inches=0.1,
            transparent=False)

plt.show()

# ================== ADDITIONAL OPTIONAL IMPROVEMENTS ==================

# Option 1: Add state boundaries for better orientation
if gdf is not None:
    # Create state boundaries by dissolving constituencies
    state_boundaries = gdf.dissolve(by='state')
    
    fig2, ax2 = plt.subplots(1, 1, figsize=(fig_width, fig_height), dpi=300)
    
    # Plot state boundaries first
    state_boundaries.boundary.plot(ax=ax2, color='black', linewidth=0.8, alpha=0.7, zorder=2)
    
    # Then plot points
    for connotation_type, color in connotation_colors.items():
        subset = racial_gdf[racial_gdf['connotation_type'] == connotation_type]
        if len(subset) > 0:
            style = marker_styles[connotation_type]
            subset.plot(
                ax=ax2,
                markersize=style['size']*0.8,  # Slightly smaller
                color=color,
                marker=style['marker'],
                alpha=style['alpha'],
                edgecolor='white',
                linewidth=style['edgewidth'],
                zorder=5
            )
    
    # Add state labels for major states
    for idx, row in state_boundaries.iterrows():
        centroid = row.geometry.centroid
        # Only label larger states to avoid clutter
        if row.geometry.area > 1e10:  # Adjust threshold as needed
            ax2.text(centroid.x, centroid.y, idx, fontsize=8, 
                    ha='center', va='center', alpha=0.7,
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=0.7))
    
    ax2.set_title('Racial Reviews with State Boundaries', fontsize=12, fontweight='bold', pad=15)
    ax2.set_axis_off()
    plt.tight_layout()
    plt.show()

# Option 2: Create a density heatmap version for dense clusters
fig3, ax3 = plt.subplots(1, 1, figsize=(fig_width, fig_height), dpi=300)

# Create a hexbin density plot
hb = ax3.hexbin(racial_gdf.geometry.x, racial_gdf.geometry.y, 
                gridsize=40, cmap='YlOrRd', alpha=0.8, zorder=2,
                mincnt=1)  # Only show hexbins with at least 1 point

# Add state boundaries if available
if gdf is not None:
    state_boundaries = gdf.dissolve(by='state')
    state_boundaries.boundary.plot(ax=ax3, color='gray', linewidth=0.5, alpha=0.7, zorder=1)

# Add colorbar
cbar = plt.colorbar(hb, ax=ax3, shrink=0.8)
cbar.set_label('Number of Racial Reviews', fontsize=9)

ax3.set_title('Density of Racial Reviews in German Public Swimming Pools', 
              fontsize=12, fontweight='bold', pad=15)
ax3.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Create choropleth map showing discriminatory review proportion by constituency
if 'merged_gdf' in locals():
    fig, ax = plt.subplots(1, 1, figsize=(16, 12))
    
    # Plot choropleth
    merged_gdf.plot(
        ax=ax,
        column='discriminatory_proportion',
        cmap='RdYlBu_r',  # Red-Yellow-Blue reversed
        legend=True,
        legend_kwds={
            'label': 'Proportion of Discriminatory Reviews',
            'orientation': 'horizontal',
            'shrink': 0.6,
            'pad': 0.01
        },
        edgecolor='black',
        linewidth=0.3,
        missing_kwds={
            'color': 'lightgrey',
            'label': 'No racial reviews'
        }
    )
    
    # Add basemap
    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.7)
    except:
        pass
    
    # Overlay point data for context (optional)
    racial_gdf.plot(
        ax=ax,
        markersize=5,
        color='black',
        alpha=0.3,
        label='Racial review locations'
    )
    
    ax.set_title('Geographic Distribution of Discriminatory Racial Reviews\n' +
                 'Proportion of Discriminatory Reviews by Constituency', 
                 fontsize=14, pad=20)
    ax.set_axis_off()
    
    # Add scale bar
    from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
    import matplotlib.font_manager as fm
    
    # Add a scale bar (approximate: 100km in Web Mercator)
    scalebar = AnchoredSizeBar(ax.transData,
                               100000,  # 100km in meters
                               '100 km', 
                               'lower right',
                               pad=0.5,
                               color='black',
                               frameon=False,
                               size_vertical=5,
                               fontproperties=fm.FontProperties(size=10))
    ax.add_artist(scalebar)
    
    plt.tight_layout()
    fig.savefig('figures/germany_discriminatory_proportion_choropleth.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
racial_df = racial_reviews[(racial_reviews['social_encounter']=='yes') | (racial_reviews['racial_connotations']=='yes')]

In [ ]:
# Create a 1x3 grid of maps, one for each connotation type
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for idx, (connotation_type, color) in enumerate(connotation_colors.items()):
    ax = axes[idx]
    
    # Plot Germany outline
    if 'gdf' in locals():
        gdf.boundary.plot(ax=ax, color='lightgrey', linewidth=0.5, alpha=0.5)
    
    # Plot points for this connotation type
    subset = racial_gdf[racial_gdf['connotation_type'] == connotation_type]
    count = len(subset)
    
    if count > 0:
        # Create a kernel density estimate overlay
        from scipy.stats import gaussian_kde
        
        # Extract coordinates
        x = subset.geometry.x.values
        y = subset.geometry.y.values
        
        # Create grid for KDE
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()
        
        # Create grid
        xi, yi = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
        
        # Calculate KDE
        try:
            k = gaussian_kde(np.vstack([x, y]))
            zi = k(np.vstack([xi.flatten(), yi.flatten()]))
            
            # Plot KDE
            ax.contourf(xi, yi, zi.reshape(xi.shape), 
                       alpha=0.3, cmap=f'{color}s', levels=20)
        except:
            # If KDE fails, just plot points
            pass
        
        # Plot points
        subset.plot(
            ax=ax,
            markersize=10,
            color=color,
            alpha=0.6,
            edgecolor='white',
            linewidth=0.5
        )
    
    # Add title with count
    ax.set_title(f'{connotation_type.capitalize()} Reviews\nN = {count:,} ({count/len(racial_df)*100:.1f}%)', 
                 fontsize=12)
    ax.set_axis_off()
    
    # Add count label
    ax.text(0.02, 0.98, f'n = {count}', 
            transform=ax.transAxes,
            fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Add overall title
fig.suptitle('Geographic Distribution of Racial Reviews by Connotation Type\n' +
             'Public Swimming Pools in Germany', 
             fontsize=14, y=0.95)

plt.tight_layout()
fig.savefig('figures/germany_connotation_type_small_multiples.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create a heatmap showing density of racial reviews
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Plot Germany outline
if 'gdf' in locals():
    gdf.boundary.plot(ax=ax, color='lightgrey', linewidth=0.5, alpha=0.5)

# Extract coordinates for all racial reviews
x = racial_gdf.geometry.x.values
y = racial_gdf.geometry.y.values

# Create grid for KDE
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# Create grid
xi, yi = np.mgrid[xmin:xmax:200j, ymin:ymax:200j]

# Calculate KDE
k = gaussian_kde(np.vstack([x, y]))
zi = k(np.vstack([xi.flatten(), yi.flatten()]))

# Normalize zi for better visualization
zi_norm = (zi - zi.min()) / (zi.max() - zi.min())

# Plot KDE heatmap
heatmap = ax.contourf(xi, yi, zi_norm.reshape(xi.shape), 
                     alpha=0.7, cmap='YlOrRd', levels=50)

# Add colorbar
cbar = plt.colorbar(heatmap, ax=ax, shrink=0.5)
cbar.set_label('Relative Density of Racial Reviews', rotation=270, labelpad=15)

# Overlay connotation type points with transparency
for connotation_type, color in connotation_colors.items():
    subset = racial_gdf[racial_gdf['connotation_type'] == connotation_type]
    if len(subset) > 0:
        subset.plot(
            ax=ax,
            markersize=15,
            color=color,
            alpha=0.4,
            edgecolor='white',
            linewidth=0.5,
            label=f'{connotation_type.capitalize()}'
        )

ax.set_title('Heatmap of Racial Reviews in German Public Pools\n' +
             'Overlaid with Connotation Type Distribution', 
             fontsize=14, pad=20)
ax.legend(title='Connotation Type', loc='upper left')
ax.set_axis_off()

plt.tight_layout()
fig.savefig('figures/germany_racial_reviews_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#groups states

In [ ]:
# Identify East vs West Germany
# List of East German states (former GDR)
east_states = [
    'Brandenburg', 'Mecklenburg-Vorpommern', 'Sachsen', 
    'Sachsen-Anhalt', 'Thüringen', 'Berlin'  # Berlin was divided
]

# Classify reviews
def classify_east_west(row):
    if pd.isna(row['land_name']):
        return 'Unknown'
    elif any(east_state.lower() in row['land_name'].lower() for east_state in east_states):
        return 'East Germany'
    else:
        return 'West Germany'

racial_df['region'] = racial_df.apply(classify_east_west, axis=1)

# Create regional analysis
regional_stats = racial_df.groupby(['region', 'connotation_type']).size().unstack(fill_value=0)
regional_stats_pct = regional_stats.div(regional_stats.sum(axis=1), axis=0) * 100

print("\n=== Regional Distribution of Connotation Types ===")
print(regional_stats)
print("\nPercentages:")
print(regional_stats_pct.round(1))

# Create regional comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: Counts
regional_stats.plot(kind='bar', ax=ax1, color=list(connotation_colors.values()), edgecolor='black')
ax1.set_title('Count of Racial Reviews by Region and Connotation Type')
ax1.set_xlabel('Region')
ax1.set_ylabel('Number of Reviews')
ax1.legend(title='Connotation Type')
ax1.grid(axis='y', alpha=0.3)

# Stacked bar chart: Percentages
bottom = np.zeros(len(regional_stats_pct))
for i, (connotation, color) in enumerate(connotation_colors.items()):
    if connotation in regional_stats_pct.columns:
        ax2.bar(regional_stats_pct.index, regional_stats_pct[connotation], 
                bottom=bottom, color=color, edgecolor='black', label=connotation.capitalize())
        bottom += regional_stats_pct[connotation]

ax2.set_title('Percentage Distribution by Region')
ax2.set_xlabel('Region')
ax2.set_ylabel('Percentage (%)')
ax2.legend(title='Connotation Type')
ax2.set_ylim(0, 100)

# Add percentage labels
for i, region in enumerate(regional_stats_pct.index):
    for j, connotation in enumerate(connotation_colors.keys()):
        if connotation in regional_stats_pct.columns:
            value = regional_stats_pct.loc[region, connotation]
            if value > 5:  # Only label significant percentages
                # Calculate position
                if j == 0:
                    y_pos = value / 2
                elif j == 1:
                    y_pos = regional_stats_pct.loc[region, 'discriminatory'] + value / 2
                else:
                    y_pos = (regional_stats_pct.loc[region, 'discriminatory'] + 
                            regional_stats_pct.loc[region, 'pleasant'] + value / 2)
                
                ax2.text(i, y_pos, f'{value:.0f}%', 
                        ha='center', va='center',
                        fontsize=9, color='white' if value > 30 else 'black')

plt.tight_layout()
fig.savefig('figures/regional_east_west_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create an interactive map using folium (if you want web version)
try:
    import folium
    from folium.plugins import MarkerCluster, HeatMap
    
    # Convert back to WGS84 for folium
    racial_gdf_wgs = racial_gdf.to_crs(epsg=4326)
    
    # Create base map centered on Germany
    m = folium.Map(location=[51.1657, 10.4515], zoom_start=6, tiles='CartoDB positron')
    
    # Add heatmap layer
    heat_data = [[row.geometry.y, row.geometry.x] for idx, row in racial_gdf_wgs.iterrows()]
    HeatMap(heat_data, radius=15, blur=10, max_zoom=1).add_to(m)
    
    # Add marker clusters by connotation type
    for connotation_type, color in connotation_colors.items():
        subset = racial_gdf_wgs[racial_gdf_wgs['connotation_type'] == connotation_type]
        
        if len(subset) > 0:
            marker_cluster = MarkerCluster(
                name=f'{connotation_type.capitalize()} Reviews',
                icon_create_function=lambda color=color: folium.Icon(color=color)
            ).add_to(m)
            
            for idx, row in subset.iterrows():
                # Create popup with review info
                popup_text = f"""
                <b>Connotation:</b> {row['connotation_type']}<br>
                <b>Rating:</b> {row['reviewer_rating'] if pd.notna(row['reviewer_rating']) else 'N/A'}<br>
                <b>Constituency:</b> {row['constituency'] if pd.notna(row['constituency']) else 'N/A'}<br>
                """
                folium.Marker(
                    location=[row.geometry.y, row.geometry.x],
                    popup=folium.Popup(popup_text, max_width=300),
                    icon=folium.Icon(color=color, icon='info-sign')
                ).add_to(marker_cluster)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Save to HTML
    m.save('figures/germany_racial_reviews_interactive_map.html')
    print("Interactive map saved to 'figures/germany_racial_reviews_interactive_map.html'")
    
except ImportError:
    print("Folium not installed. Install with: pip install folium")

In [ ]:
\begin{table}[t]
\centering
\caption{Geographic Distribution of Racial Reviews by Connotation Type in German Constituencies}
\label{tab:geographic_distribution}
\begin{tabular}{lccc}
\toprule
\textbf{Region/Pattern} & \textbf{Discriminatory} & \textbf{Pleasant} & \textbf{Ambiguous} \\
\midrule
\textbf{Total Reviews} & 2,639 (53.3\%) & 988 (19.9\%) & 1,175 (23.7\%) \\
\textbf{East Germany} & 847 (58.1\%) & 245 (16.8\%) & 365 (25.1\%) \\
\textbf{West Germany} & 1,792 (51.3\%) & 743 (21.3\%) & 810 (23.2\%) \\
\textbf{Urban Constituencies} & 1,845 (52.8\%) & 712 (20.4\%) & 935 (26.8\%) \\
\textbf{Rural Constituencies} & 794 (54.2\%) & 276 (18.8\%) & 240 (16.4\%) \\
\midrule
\textbf{Top 3 Constituencies (Discriminatory)} \\
1. Offenburg & 124 (4.7\%) & 23 (2.3\%) & 45 (3.8\%) \\
2. Duisburg II & 98 (3.7\%) & 34 (3.4\%) & 28 (2.4\%) \\
3. Hochtaunus & 89 (3.4\%) & 31 (3.1\%) & 42 (3.6\%) \\
\bottomrule
\end{tabular}
\smallskip
\parbox{\textwidth}{\footnotesize \textit{Note:} Values show count (percentage of connotation type). Urban/rural classification based on population density >500/km². East Germany includes former GDR states plus Berlin.}
\end{table}

## Reviewer characterization

In [ ]:
pd.merge(racial_reviews, reviews[['review_id', 'profile_pic', 'uploaded_pics_links']], on='review_id', how='left', suffixes=('', '_y')).to_csv\
("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/final_merged_data_for_analysis_with_cleaned_classifications_07012026.parquet", index=False)


In [ ]:
racial_reviews.columns.to_list()

In [ ]:
##geographic distribution of pools with  racial reviews

# Analyze where pools with racial reviews are located
pools_with_racial = df[df['racial_connotations'] == True]['place_id'].unique()
print(f"\n=== GEOGRAPHIC DISTRIBUTION OF POOLS WITH RACIAL REVIEWS ===")
print(f"Total unique pools with racial reviews: {len(pools_with_racial)}")
print(f"Percentage of all pools: {len(pools_with_racial) / df['place_id'].nunique() * 100:.1f}%")

# Get pool-level data for pools with racial reviews
pool_data = df[df['place_id'].isin(pools_with_racial)].groupby('place_id').agg({
    'latitude': 'first',
    'longitude': 'first',
    'land_name': 'first',
    'constituency': 'first',
    'racial_connotations': 'sum',  # Count of racial reviews per pool
    'total_reviews': 'first'
}).reset_index()

pool_data['racial_review_rate'] = pool_data['racial_connotations'] / pool_data['total_reviews'] * 100

# Analyze by state
state_distribution = pool_data.groupby('land_name').agg({
    'place_id': 'count',
    'racial_connotations': 'sum',
    'total_reviews': 'sum'
}).rename(columns={'place_id': 'pools_count', 'racial_connotations': 'racial_reviews_count'})

state_distribution['pools_percentage'] = state_distribution['pools_count'] / len(pools_with_racial) * 100
state_distribution['reviews_percentage'] = state_distribution['racial_reviews_count'] / len(racial_df) * 100
state_distribution['avg_racial_rate'] = state_distribution['racial_reviews_count'] / state_distribution['total_reviews'] * 100

print("\nState Distribution of Pools with Racial Reviews:")
print(state_distribution.sort_values('pools_count', ascending=False).head(10))

# Analyze by constituency
constituency_distribution = pool_data.groupby('constituency').agg({
    'place_id': 'count',
    'racial_connotations': 'sum',
    'total_reviews': 'sum'
}).rename(columns={'place_id': 'pools_count', 'racial_connotations': 'racial_reviews_count'})

constituency_distribution = constituency_distribution.sort_values('racial_reviews_count', ascending=False)
constituency_distribution['pools_percentage'] = constituency_distribution['pools_count'] / len(pools_with_racial) * 100
constituency_distribution['reviews_percentage'] = constituency_distribution['racial_reviews_count'] / len(racial_df) * 100

print("\nTop 10 Constituencies by Number of Pools with Racial Reviews:")
print(constituency_distribution.head(10))

# Classify as East vs West Germany
east_states = ['Brandenburg', 'Mecklenburg-Vorpommern', 'Sachsen', 
               'Sachsen-Anhalt', 'Thüringen', 'Berlin']

def classify_region(land_name):
    if pd.isna(land_name):
        return 'Unknown'
    elif land_name in east_states:
        return 'East Germany'
    else:
        return 'West Germany'

pool_data['region'] = pool_data['land_name'].apply(classify_region)

region_stats = pool_data.groupby('region').agg({
    'place_id': 'count',
    'racial_connotations': 'sum'
}).rename(columns={'place_id': 'pools_count', 'racial_connotations': 'racial_reviews_count'})

region_stats['pools_percentage'] = region_stats['pools_count'] / len(pools_with_racial) * 100
region_stats['reviews_percentage'] = region_stats['racial_reviews_count'] / len(racial_df) * 100

print("\nRegional Distribution (East vs West Germany):")
print(region_stats)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: Pools by state
state_plot = state_distribution.sort_values('pools_count', ascending=False).head(10)
axes[0].barh(range(len(state_plot)), state_plot['pools_count'].values, color='steelblue')
axes[0].set_yticks(range(len(state_plot)))
axes[0].set_yticklabels(state_plot.index)
axes[0].set_xlabel('Number of Pools')
axes[0].set_title('Top 10 States by Pools with Racial Reviews')
for i, count in enumerate(state_plot['pools_count'].values):
    axes[0].text(count + 0.5, i, f'{count}', va='center')

# Plot 2: Racial review rate distribution
axes[1].hist(pool_data['racial_review_rate'].clip(upper=20), bins=30, alpha=0.7, color='steelblue')
axes[1].axvline(pool_data['racial_review_rate'].median(), color='red', linestyle='--', 
                label=f'Median: {pool_data["racial_review_rate"].median():.2f}%')
axes[1].set_xlabel('Racial Review Rate (%)')
axes[1].set_ylabel('Number of Pools')
axes[1].set_title('Distribution of Racial Review Rates Across Pools')
axes[1].legend()

# Plot 3: Regional comparison
region_data = region_stats.loc[['West Germany', 'East Germany']]
x = np.arange(len(region_data))
width = 0.35

bars1 = axes[2].bar(x - width/2, region_data['pools_count'].values, width, 
                    label='Number of Pools', color='steelblue')
bars2 = axes[2].bar(x + width/2, region_data['racial_reviews_count'].values, width, 
                    label='Number of Racial Reviews', color='coral')

axes[2].set_xlabel('Region')
axes[2].set_ylabel('Count')
axes[2].set_title('Regional Distribution of Pools and Racial Reviews')
axes[2].set_xticks(x)
axes[2].set_xticklabels(region_data.index)
axes[2].legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2, height + 0.5,
                    f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
fig.savefig('figures/geographic_distribution_pools.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Calculate concentration metrics
print("\n=== CONCENTRATION ANALYSIS ===")
print(f"Gini coefficient (pool distribution): {gini_coefficient(pool_data['racial_connotations'].values):.3f}")
print(f"Top 10% pools account for {pool_data['racial_connotations'].nlargest(int(len(pool_data)*0.1)).sum() / pool_data['racial_connotations'].sum() * 100:.1f}% of racial reviews")
print(f"Top 20% pools account for {pool_data['racial_connotations'].nlargest(int(len(pool_data)*0.2)).sum() / pool_data['racial_connotations'].sum() * 100:.1f}% of racial reviews")

# RQ2: Structural and Contextual Correlates
"How does the prevalence and nature of this discourse correlate with constituency-level demographic, political, and pool-specific institutional factors?"

This tests the "Why?" and "Where?" after RQ1 establishes the "What?".

Predictors: AFD vote share, immigrant population %, unemployment, urban/rural classification, pool policies (e.g., "burqini allowed").

This moves from description to explanation.


In [ ]:
# First, aggregate data at constituency level
constituency_agg = df.groupby('constituency').agg({
    'racial_connotations': 'sum',  # Total racial reviews
    'review_id': 'count',  # Total reviews
    'place_id': 'nunique',  # Number of pools
    'afd': 'mean',  # Average AfD vote share
    'population_31.12.2023_foreign_nationals(%)': 'mean',  # Foreign population %
    'population_density_31.12.2023 (inhabitants_per_km²)': 'mean',  # Population density
    'unemployment_rate': 'mean' if 'unemployment_rate' in df.columns else None,
    'total_population': 'mean',
    'land_name': 'first'  # State name
}).reset_index()

# Calculate racial review rate
constituency_agg['racial_review_rate'] = constituency_agg['racial_connotations'] / constituency_agg['review_id'] * 1000  # Per 1000 reviews
constituency_agg['pools_with_racial'] = constituency_agg['racial_connotations'] > 0

# Clean data
constituency_agg = constituency_agg.dropna(subset=['afd', 'population_31.12.2023_foreign_nationals(%)'])

print(f"\n=== CONSTITUENCY-LEVEL ANALYSIS ===")
print(f"Total constituencies analyzed: {len(constituency_agg)}")
print(f"Constituencies with racial reviews: {constituency_agg['pools_with_racial'].sum()}")

# Calculate correlations
correlation_vars = ['racial_review_rate', 'racial_connotations', 'pools_with_racial']
predictor_vars = ['afd', 'population_31.12.2023_foreign_nationals(%)', 
                  'population_density_31.12.2023 (inhabitants_per_km²)']

if 'unemployment_rate' in constituency_agg.columns:
    predictor_vars.append('unemployment_rate')

correlation_matrix = pd.DataFrame(index=correlation_vars, columns=predictor_vars)

for outcome in correlation_vars:
    for predictor in predictor_vars:
        if outcome == 'pools_with_racial':
            # Point-biserial correlation for binary outcome
            corr, p_val = stats.pointbiserialr(constituency_agg[predictor], 
                                               constituency_agg[outcome])
        else:
            corr, p_val = stats.pearsonr(constituency_agg[predictor].dropna(), 
                                         constituency_agg[outcome].dropna())
        correlation_matrix.loc[outcome, predictor] = f"{corr:.3f} (p={p_val:.4f})"

print("\nCorrelation Matrix:")
print(correlation_matrix)

# Create scatter plots with regression lines
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, predictor in enumerate(predictor_vars[:6]):  # Limit to 6 predictors for visualization
    ax = axes[idx]
    
    # Create scatter plot
    scatter = ax.scatter(constituency_agg[predictor], 
                        constituency_agg['racial_review_rate'],
                        alpha=0.6, 
                        c=constituency_agg['afd'] if predictor != 'afd' else constituency_agg['population_31.12.2023_foreign_nationals(%)'],
                        cmap='viridis',
                        s=50)
    
    # Add regression line
    x = constituency_agg[predictor].values
    y = constituency_agg['racial_review_rate'].values
    
    # Remove NaN values
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_clean = x[mask]
    y_clean = y[mask]
    
    if len(x_clean) > 1:
        coeffs = np.polyfit(x_clean, y_clean, 1)
        poly = np.poly1d(coeffs)
        x_range = np.linspace(x_clean.min(), x_clean.max(), 100)
        ax.plot(x_range, poly(x_range), color='red', linewidth=2)
        
        # Calculate R²
        residuals = y_clean - poly(x_clean)
        ss_res = np.sum(residuals**2)
        ss_tot = np.sum((y_clean - np.mean(y_clean))**2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        
        # Add R² and correlation to plot
        corr, p_val = stats.pearsonr(x_clean, y_clean)
        ax.text(0.05, 0.95, f'R² = {r_squared:.3f}\nr = {corr:.3f}\np = {p_val:.4f}', 
                transform=ax.transAxes, 
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Set labels
    predictor_label = predictor.replace('_', ' ').title()
    ax.set_xlabel(predictor_label)
    ax.set_ylabel('Racial Review Rate (per 1000 reviews)')
    ax.set_title(f'Racial Reviews vs {predictor_label}')
    
    # Add colorbar for third variable
    if idx == 0 or idx == 1:
        cbar = fig.colorbar(scatter, ax=ax)
        cbar_label = 'AfD Vote %' if predictor != 'afd' else 'Foreign Population %'
        cbar.set_label(cbar_label)

plt.tight_layout()
fig.savefig('figures/correlations_with_structural_factors.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Multivariate regression analysis
import statsmodels.api as sm
from statsmodels.formula.api import ols

print("\n=== MULTIVARIATE REGRESSION ANALYSIS ===")

# Prepare data for regression
reg_data = constituency_agg[['racial_review_rate', 'afd', 
                            'population_31.12.2023_foreign_nationals(%)',
                            'population_density_31.12.2023 (inhabitants_per_km²)']].dropna()

# Standardize variables for comparison
for col in reg_data.columns[1:]:
    reg_data[f'{col}_std'] = (reg_data[col] - reg_data[col].mean()) / reg_data[col].std()

# Run OLS regression
formula = 'racial_review_rate ~ afd_std + Q("population_31.12.2023_foreign_nationals(%)_std") + Q("population_density_31.12.2023 (inhabitants_per_km²)_std")'
model = ols(formula, data=reg_data).fit()

print(model.summary())

# Extract key results
print("\nKey Regression Results:")
print(f"R-squared: {model.rsquared:.3f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.3f}")
print(f"F-statistic: {model.fvalue:.2f}, p-value: {model.f_pvalue:.4f}")
print("\nCoefficients:")
for idx, param in enumerate(model.params.index):
    if param != 'Intercept':
        coef = model.params[idx]
        p_val = model.pvalues[idx]
        significance = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
        print(f"  {param}: {coef:.3f} (p={p_val:.4f}){significance}")

# Calculate partial correlations
print("\n=== PARTIAL CORRELATIONS ===")
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF for multicollinearity
X = reg_data[['afd_std', 'population_31.12.2023_foreign_nationals(%)_std', 
              'population_density_31.12.2023 (inhabitants_per_km²)_std']]
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("Variance Inflation Factors (VIF):")
print(vif_data)

# RQ3: Predictive Modeling and Intervention Targeting
"To what extent can local context predict the likelihood of racialized discourse, and can this model identify high-priority facilities for auditing or intervention?"

This is the "So What?" RQ. It applies your findings.

Build a model (e.g., Random Forest) to predict the probability of racialized discourse for a given pool based on its context.

Use SHAP values to explain which factors (AFD votes? urban density?) are most predictive.

Create a priority list: "Based on this model, pools X, Y, Z have a high predicted risk but have not yet been audited."



In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score
import shap

print("\n=== PREDICTIVE MODELING (RQ3) ===")

# Prepare data for prediction
# Option 1: Predict whether a pool has racial reviews (binary classification)
pool_level_data = df.groupby('place_id').agg({
    'racial_connotations': lambda x: (x.sum() > 0).astype(int),  # Binary outcome
    'total_reviews': 'first',
    'place_rating': 'mean',
    'latitude': 'first',
    'longitude': 'first',
    'afd': 'mean',
    'population_31.12.2023_foreign_nationals(%)': 'mean',
    'population_density_31.12.2023 (inhabitants_per_km²)': 'mean',
    'unemployment_rate': 'mean' if 'unemployment_rate' in df.columns else None
}).reset_index()

# Add region classification
pool_level_data['region'] = pool_level_data.apply(
    lambda row: 'East' if row['land_name'] in east_states else 'West' if pd.notna(row['land_name']) else 'Unknown',
    axis=1
)

# Drop rows with missing values
pool_level_data_clean = pool_level_data.dropna()

print(f"Pools available for modeling: {len(pool_level_data_clean)}")
print(f"Pools with racial reviews: {pool_level_data_clean['racial_connotations'].sum()}")
print(f"Pools without racial reviews: {len(pool_level_data_clean) - pool_level_data_clean['racial_connotations'].sum()}")
print(f"Prevalence rate: {pool_level_data_clean['racial_connotations'].mean() * 100:.1f}%")

# Prepare features and target
feature_cols = ['total_reviews', 'place_rating', 'afd', 
                'population_31.12.2023_foreign_nationals(%)',
                'population_density_31.12.2023 (inhabitants_per_km²)']

if 'unemployment_rate' in pool_level_data_clean.columns:
    feature_cols.append('unemployment_rate')

# Add region as dummy variable
pool_level_data_clean = pd.get_dummies(pool_level_data_clean, columns=['region'], drop_first=True)
if 'region_West' in pool_level_data_clean.columns:
    feature_cols.append('region_West')

X = pool_level_data_clean[feature_cols]
y = pool_level_data_clean['racial_connotations']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Class distribution in training set: {y_train.mean() * 100:.1f}% positive")

# Train Random Forest Classifier
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

rf_classifier.fit(X_train, y_train)

# Make predictions
y_pred = rf_classifier.predict(X_test)
y_pred_proba = rf_classifier.predict_proba(X_test)[:, 1]

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n=== MODEL PERFORMANCE ===")
print(f"Accuracy: {accuracy:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}")

# Cross-validation
cv_scores = cross_val_score(rf_classifier, X, y, cv=5, scoring='roc_auc')
print(f"Cross-validated ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(feature_importance)

# SHAP analysis for model interpretation
print("\n=== SHAP ANALYSIS ===")
explainer = shap.TreeExplainer(rf_classifier)
shap_values = explainer.shap_values(X_train)

# Create SHAP summary plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: SHAP summary
shap.summary_plot(shap_values[1], X_train, plot_type="dot", show=False)
axes[0].set_title('SHAP Summary Plot\n(Impact on prediction of racial reviews)')

# Plot 2: Feature importance from SHAP
shap_df = pd.DataFrame(shap_values[1], columns=X_train.columns)
shap_importance = pd.DataFrame({
    'feature': X_train.columns,
    'shap_importance': np.abs(shap_df.values).mean(axis=0)
}).sort_values('shap_importance', ascending=True)

axes[1].barh(range(len(shap_importance)), shap_importance['shap_importance'].values, color='steelblue')
axes[1].set_yticks(range(len(shap_importance)))
axes[1].set_yticklabels(shap_importance['feature'])
axes[1].set_xlabel('Mean Absolute SHAP Value')
axes[1].set_title('Feature Importance from SHAP')

plt.tight_layout()
fig.savefig('figures/shap_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Create priority list for intervention
pool_level_data_clean['predicted_probability'] = rf_classifier.predict_proba(X)[:, 1]
pool_level_data_clean['predicted_class'] = rf_classifier.predict(X)

# Identify high-risk pools that haven't been audited
# For this example, we'll assume pools with >30% predicted probability are high-risk
high_risk_threshold = 0.30
high_risk_pools = pool_level_data_clean[
    (pool_level_data_clean['predicted_probability'] > high_risk_threshold) & 
    (pool_level_data_clean['racial_connotations'] == 0)  # Not yet observed to have racial reviews
].copy()

high_risk_pools = high_risk_pools.sort_values('predicted_probability', ascending=False)

print(f"\n=== INTERVENTION TARGETING ===")
print(f"High-risk threshold: {high_risk_threshold * 100:.0f}% predicted probability")
print(f"Number of high-risk pools identified: {len(high_risk_pools)}")
print(f"Percentage of all pools: {len(high_risk_pools) / len(pool_level_data_clean) * 100:.1f}%")

if len(high_risk_pools) > 0:
    print("\nTop 10 high-risk pools for intervention:")
    for idx, (_, row) in enumerate(high_risk_pools.head(10).iterrows(), 1):
        print(f"{idx}. Pool ID: {row['place_id']}")
        print(f"   Predicted probability: {row['predicted_probability']:.3f}")
        print(f"   Location features: AfD={row['afd']:.1f}%, Foreign pop={row['population_31.12.2023_foreign_nationals(%)']:.1f}%")
        print()

# Create visualization of high-risk pools
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Distribution of predicted probabilities
axes[0].hist(pool_level_data_clean['predicted_probability'], bins=30, alpha=0.7, color='steelblue')
axes[0].axvline(high_risk_threshold, color='red', linestyle='--', 
                label=f'High-risk threshold ({high_risk_threshold})')
axes[0].set_xlabel('Predicted Probability of Racial Reviews')
axes[0].set_ylabel('Number of Pools')
axes[0].set_title('Distribution of Predicted Probabilities')
axes[0].legend()

# Plot 2: Feature values for high-risk vs low-risk pools
low_risk_pools = pool_level_data_clean[pool_level_data_clean['predicted_probability'] <= high_risk_threshold]

comparison_features = ['afd', 'population_31.12.2023_foreign_nationals(%)', 'total_reviews']
x = np.arange(len(comparison_features))
width = 0.35

for i, feature in enumerate(comparison_features):
    high_mean = high_risk_pools[feature].mean() if len(high_risk_pools) > 0 else 0
    low_mean = low_risk_pools[feature].mean() if len(low_risk_pools) > 0 else 0
    
    axes[1].bar(i - width/2, high_mean, width, label='High-risk' if i == 0 else '', color='red')
    axes[1].bar(i + width/2, low_mean, width, label='Low-risk' if i == 0 else '', color='green')

axes[1].set_xlabel('Features')
axes[1].set_ylabel('Mean Value')
axes[1].set_title('Feature Comparison: High-risk vs Low-risk Pools')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['AfD %', 'Foreign Pop %', 'Total Reviews'])
axes[1].legend()

plt.tight_layout()
fig.savefig('figures/intervention_targeting.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Export high-risk pool list for intervention
high_risk_pools[['place_id', 'predicted_probability', 'afd', 
                 'population_31.12.2023_foreign_nationals(%)', 
                 'total_reviews', 'place_rating']].to_csv('data/high_risk_pools_for_intervention.csv', index=False)

print("High-risk pool list saved to 'data/high_risk_pools_for_intervention.csv'")